# 08. QCHS Personalized Stage-1 Retrieval and Winner Locking — Herbal Supplements

This notebook imports the locked Notebook 07 query-only winner as a fixed baseline and evaluates three personalized Stage-1 variants: Profile Sparse QCHS, Profile Hybrid QCHS, and Profile Full QCHS. All methods preserve the same query universe, item universe, and exact top-1,000 candidate budget.

Query-conditioned history selection (QCHS) uses only training-safe prior items. Prior-item selection is driven by positive functional alignment with the current query; Brand cannot drive selection because it is absent from the synthetic query. Once an item is selected, its profile-safe catalog-functional and Brand facets may enter the user profile. Frozen review-derived item signals remain available in the item representation but are excluded from user-profile evidence.

The personalized variants fuse the unchanged query-only winner with profile-expanded sparse, dense, or graph channels. If a case is cold or has no positively aligned profile-safe prior item, the personalized condition copies the query-only candidate identifiers, order, and scores exactly. These fallback cases remain in the full evaluation population as no-op personalization cases.

The stored execution contains 553 QCHS-active cases and 1,415 exact fallbacks among 1,968 cases. The selected personalized winner is Profile Sparse QCHS. Its QCHS-active coverage is 28.1% overall: 0% in cold, 25.8% in weak, and 58.5% in strong cases.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q pyarrow rank-bm25 sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 131.9 MB/s eta 0:00:00


In [3]:
# ==== Load Libraries ====
from collections import Counter, defaultdict
from pathlib import Path
import json
import math
import re
import time

import faiss
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


In [4]:
# ==== Define Inputs, QCHS Policies, Fusion Weights, and Outputs ====
CATEGORY_ID = 'herbal'
CATEGORY_LABEL = 'Herbal Supplements'
PROJECT_ROOT = Path('/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements')

QUERY_CACHE_PATH = PROJECT_ROOT / 'outputs/query_cache/herbal_query_cache.parquet'
QUERY_CONTRACT_PATH = PROJECT_ROOT / 'outputs/query_cache/herbal_query_generation_config.json'
PRIOR_HISTORY_PATH = PROJECT_ROOT / 'data/processed/user_sampling/herbal_user_prior_review_history_training.parquet'
ITEM_DOCS_PATH = PROJECT_ROOT / 'data/processed/items/item_docs_herbal.parquet'
ITEM_FACETS_PATH = PROJECT_ROOT / 'data/processed/items/items_facets_herbal.parquet'
RETRIEVAL_ARTIFACT_MANIFEST_PATH = PROJECT_ROOT / 'data/processed/items/retrieval_artifact_manifest_herbal.json'
STAGE1_SELECTION_PATH = PROJECT_ROOT / 'outputs/stage1_query_retrieval_selection/stage1_method_selection_herbal.csv'
STAGE1_MANIFEST_PATH = PROJECT_ROOT / 'outputs/stage1_query_retrieval_selection/stage1_run_manifest_herbal.json'
STAGE1_WINNER_MANIFEST_PATH = PROJECT_ROOT / f'outputs/stage1_query_retrieval_selection/stage1_query_only_winner_{CATEGORY_ID}.json'

OUTPUT_DIR = PROJECT_ROOT / 'outputs/stage1_personalized_retrieval'
CANDIDATE_LISTS_PATH = OUTPUT_DIR / 'personalized_retrieval_candidate_lists_herbal.parquet'
PER_QUERY_METRICS_PATH = OUTPUT_DIR / 'personalized_retrieval_per_query_metrics_herbal.parquet'
PROFILE_DIAGNOSTICS_PATH = OUTPUT_DIR / 'personalized_retrieval_user_profile_diagnostics_herbal.csv'
MANIFEST_PATH = OUTPUT_DIR / 'personalized_retrieval_manifest_herbal.json'

METHOD_SELECTION_PATH = OUTPUT_DIR / f"personalized_retrieval_method_selection_{CATEGORY_ID}.csv"
WINNER_MANIFEST_PATH = OUTPUT_DIR / f"personalized_retrieval_winner_{CATEGORY_ID}.json"
WINNER_CONTRACT_VERSION = "stage1_personalized_retrieval_winner_v1"
PERSONALIZED_WINNER_METHOD_OVERRIDE = None

ACTIVE_QUERY_COLUMN = "query"
COMPATIBILITY_QUERY_ALIAS = 'query_C'
PRIOR_HAS_TARGET_PARENT_ASIN = True
DENSE_TEXT_COLUMN = "dense_text"
SPARSE_TEXT_COLUMN = "sparse_text"
BRAND_TEXT_COLUMN = "brand_facet_text"
PROFILE_SAFE_TEXT_COLUMN = "profile_safe_facet_text"
ITEM_EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
PROFILE_EVIDENCE_SCOPE = "leakage_safe_pre_target_user_prior_items_with_metadata_functional_and_brand_facets"

EXPECTED_QUERY_ROWS = None
EXPECTED_REGIME_COUNTS = None
REGIME_ORDER = ['cold', 'weak', 'strong']
QUERY_PASSTHROUGH_COLUMNS = ['target_rank_desc']

MAX_RETRIEVAL_K = 1000
EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]
MAX_QCHS_PRIOR_ITEMS = 12
MAX_ANCHOR_PHRASES = 16
MAX_ANCHOR_PHRASES_PER_ROLE = 4
QUERY_REPEAT = 3
RRF_K = 60
PROFILE_SPARSE_WEIGHTS = [1.0, 0.8]
PROFILE_HYBRID_WEIGHTS = [1.0, 0.5, 0.7]
PROFILE_FULL_WEIGHTS = [1.0, 0.4, 0.6, 0.6]
ATTENTION_TEMPERATURE = 0.25
GRAPH_ITEMS_PER_FACET_LIMIT = 5000
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
ITEM_EMBEDDING_BATCH_SIZE = 128
QUERY_EMBEDDING_BATCH_SIZE = 256
RANDOM_SEED = 42

FUNCTIONAL_ROLES = {'category_or_product_type',
 'claim_constraint',
 'form_texture',
 'ingredient_or_composition',
 'need_benefit_concern',
 'sensory',
 'target_context'}
PROFILE_ROLES = set(FUNCTIONAL_ROLES) | {"brand"}
TOKEN_EQUIVALENCE = {'berries': 'berry',
 'berry': 'berry',
 'calm': 'calm',
 'capsule': 'capsule',
 'capsules': 'capsule',
 'digestion': 'digestion',
 'digestive': 'digestion',
 'drop': 'drop',
 'drops': 'drop',
 'energy': 'energy',
 'extract': 'extract',
 'extracts': 'extract',
 'focus': 'focus',
 'gummies': 'gummy',
 'gummy': 'gummy',
 'herb': 'herbal',
 'herbal': 'herbal',
 'herbs': 'herbal',
 'immune': 'immune',
 'immunity': 'immune',
 'joint': 'joint',
 'joints': 'joint',
 'leaf': 'leaf',
 'leaves': 'leaf',
 'liquid': 'liquid',
 'liquids': 'liquid',
 'mushroom': 'mushroom',
 'mushrooms': 'mushroom',
 'powder': 'powder',
 'powders': 'powder',
 'relax': 'relaxation',
 'relaxation': 'relaxation',
 'relaxing': 'relaxation',
 'root': 'root',
 'roots': 'root',
 'sleep': 'sleep',
 'softgel': 'softgel',
 'softgels': 'softgel',
 'stress': 'stress',
 'supplement': 'supplement',
 'supplements': 'supplement',
 'support': 'support',
 'supported': 'support',
 'supporting': 'support',
 'supports': 'support',
 'tablet': 'tablet',
 'tablets': 'tablet',
 'tea': 'tea',
 'teas': 'tea',
 'tincture': 'tincture',
 'tinctures': 'tincture'}
LINGUISTIC_STOPWORDS = {'a',
 'an',
 'and',
 'are',
 'as',
 'at',
 'be',
 'by',
 'for',
 'from',
 'in',
 'into',
 'is',
 'it',
 'of',
 'on',
 'or',
 'that',
 'the',
 'this',
 'to',
 'with',
 'without',
 'you',
 'your'}
GENERIC_ANCHOR_TOKENS = {'blend',
 'daily',
 'formula',
 'herbal',
 'item',
 'items',
 'natural',
 'product',
 'products',
 'routine',
 'solution',
 'solutions',
 'supplement',
 'supplements',
 'support',
 'wellness'}
GENERIC_UTILITY_TOKENS = {'blend',
 'care',
 'complex',
 'formula',
 'natural',
 'product',
 'products',
 'routine',
 'solution',
 'solutions',
 'support',
 'wellness'}
ATTENTION_STOPWORDS = {'a',
 'an',
 'and',
 'are',
 'as',
 'at',
 'be',
 'blend',
 'boost',
 'by',
 'care',
 'complex',
 'dietary',
 'for',
 'formula',
 'from',
 'health',
 'help',
 'helps',
 'herbal',
 'in',
 'into',
 'is',
 'it',
 'its',
 'of',
 'on',
 'or',
 'over',
 'product',
 'promote',
 'promotes',
 'routine',
 'solution',
 'supplement',
 'supplements',
 'support',
 'supports',
 'that',
 'the',
 'these',
 'this',
 'those',
 'to',
 'under',
 'wellness',
 'with',
 'without'}

PROFILE_METHODS = [
    "profile_sparse_qchs",
    "profile_hybrid_qchs",
    "profile_full_qchs",
]
PROFILE_METHOD_LABELS = {
    "profile_sparse_qchs": "Profile Sparse QCHS",
    "profile_hybrid_qchs": "Profile Hybrid QCHS",
    "profile_full_qchs": "Profile Full QCHS",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", QUERY_CACHE_PATH)
print("Input:", PRIOR_HISTORY_PATH)
print("Input winner contract:", STAGE1_WINNER_MANIFEST_PATH)
print("Output:", OUTPUT_DIR)


Input: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/query_cache/herbal_query_cache.parquet
Input: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/data/processed/user_sampling/herbal_user_prior_review_history_training.parquet
Input winner contract: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_query_retrieval_selection/stage1_query_only_winner_herbal.json
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval


In [5]:
# ==== Define Retrieval, Profile, Fusion, and Metric Helpers ====
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def normalize_entity_value(value):
    text = normalize_space(value).lower()
    text = re.sub(r"[^a-z0-9??-?R\s_\-+/]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def require_columns(frame, required, frame_name):
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{frame_name} missing required columns: {missing}")


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def canonical_tokens(value):
    text = normalize_space(value).lower().replace("_", " ").replace("-", " ").replace("/", " ")
    tokens = re.findall(r"[a-z0-9']+", text)
    return [TOKEN_EQUIVALENCE.get(token, token) for token in tokens]


def tokenize_sparse_document(value):
    return [token for token in canonical_tokens(value) if token not in LINGUISTIC_STOPWORDS]


def tokenize_sparse_query(value):
    stopwords = LINGUISTIC_STOPWORDS | GENERIC_ANCHOR_TOKENS
    return [token for token in canonical_tokens(value) if token not in stopwords]


def stable_unique(values):
    return list(dict.fromkeys(values))


def bm25_top_exact(bm25, query_tokens, top_k):
    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float64)
    if len(scores) == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float64)
    if not np.isfinite(scores).all():
        raise RuntimeError("BM25 scores contain non-finite values.")
    k_eff = min(int(top_k), len(scores))
    stable_position = np.arange(len(scores), dtype=np.int64)
    order = np.lexsort((stable_position, -scores))
    selected = order[:k_eff]
    return selected.astype(np.int64), scores[selected].astype(np.float64)


def top_exact_item_ids(bm25, query_tokens, item_ids, top_k):
    selected, _ = bm25_top_exact(bm25, query_tokens, top_k)
    return [item_ids[int(index)] for index in selected]


def validate_bm25_top_exact_helper():
    class StaticBM25:
        def __init__(self, scores):
            self.scores = np.asarray(scores, dtype=np.float64)
        def get_scores(self, query_tokens):
            return self.scores

    cases = [
        ([0.0, 0.0, 0.0, 0.0], 3),
        ([3.0, 2.0, 0.0, 0.0], 3),
        ([3.0, 2.0, 1.0], 3),
        ([5.0, 4.0, 3.0, 2.0], 3),
        ([0.0, 0.0, 0.0, 0.0, 0.0], 4),
        ([2.0, 0.0, -1.0, -3.0], 4),
        ([1.0, 0.0], 5),
    ]
    for scores, top_k in cases:
        first_idx, first_scores = bm25_top_exact(StaticBM25(scores), ["query"], top_k)
        second_idx, second_scores = bm25_top_exact(StaticBM25(scores), ["query"], top_k)
        expected_k = min(int(top_k), len(scores))
        expected_order = np.lexsort((np.arange(len(scores), dtype=np.int64), -np.asarray(scores, dtype=np.float64)))[:expected_k]
        if not np.array_equal(first_idx, second_idx) or not np.array_equal(first_scores, second_scores):
            raise RuntimeError("bm25_top_exact is not deterministic.")
        if len(first_idx) != expected_k:
            raise RuntimeError("bm25_top_exact returned the wrong candidate count.")
        if len(set(first_idx.tolist())) != len(first_idx):
            raise RuntimeError("bm25_top_exact returned duplicate indexes.")
        if not np.array_equal(first_idx, expected_order):
            raise RuntimeError("bm25_top_exact ordering mismatch.")
    print("Validation: BM25 exact-K helper tests passed")


validate_bm25_top_exact_helper()


def top_items_from_score_map(score_map, top_k):
    return [
        item_id
        for item_id, _ in sorted(score_map.items(), key=lambda value: (-value[1], value[0]))[:top_k]
    ]


def rrf_fuse(source_lists, weights, top_k):
    scores = defaultdict(float)
    for source_items, weight in zip(source_lists, weights):
        for rank, item_id in enumerate(source_items, start=1):
            scores[item_id] += float(weight) / (RRF_K + rank)
    ranked = sorted(scores, key=lambda item_id: (-scores[item_id], item_id))[:top_k]
    return ranked, [float(scores[item_id]) for item_id in ranked]


def rank_metrics(rank, k):
    hit = int(0 < rank <= k)
    ndcg = 1.0 / math.log2(rank + 1) if hit else 0.0
    mrr = 1.0 / rank if hit else 0.0
    return hit, ndcg, mrr


def weighted_overlap(query_terms_value, item_terms, token_idf):
    query_term_set = set(query_terms_value)
    item_term_set = set(item_terms)
    if not query_term_set:
        return 0.0
    denominator = sum(token_idf.get(token, 1.0) for token in query_term_set)
    numerator = sum(token_idf.get(token, 1.0) for token in query_term_set & item_term_set)
    return float(numerator / max(denominator, 1e-12))


def softmax_weights(scores):
    values = np.asarray(scores, dtype=float)
    if len(values) == 0:
        return np.array([], dtype=float)
    shifted = values / ATTENTION_TEMPERATURE
    shifted -= shifted.max()
    weights = np.exp(shifted)
    return weights / weights.sum()


def normalized_entropy(weights):
    values = np.asarray(weights, dtype=float)
    values = values[values > 0]
    if len(values) <= 1:
        return 0.0
    entropy = -float(np.sum(values * np.log(values)))
    return entropy / math.log(len(values))


def prior_depth_bin(count):
    count = int(count)
    if count == 0:
        return "0"
    if count <= 2:
        return "1-2"
    if count <= 4:
        return "3-4"
    if count <= 9:
        return "5-9"
    return "10+"


def contains_token_sequence(sequence, subsequence):
    if not subsequence or len(subsequence) > len(sequence):
        return False
    width = len(subsequence)
    return any(tuple(sequence[start:start + width]) == tuple(subsequence) for start in range(len(sequence) - width + 1))


def query_terms(text):
    return stable_unique(
        token for token in canonical_tokens(text)
        if token not in ATTENTION_STOPWORDS and len(token) > 1
    )


Validation: BM25 exact-K helper tests passed


In [6]:
# ==== Load the Query-Only Winner and Validate Upstream Contracts ====
required_paths = [
    QUERY_CACHE_PATH,
    QUERY_CONTRACT_PATH,
    PRIOR_HISTORY_PATH,
    ITEM_DOCS_PATH,
    ITEM_FACETS_PATH,
    RETRIEVAL_ARTIFACT_MANIFEST_PATH,
    STAGE1_SELECTION_PATH,
    STAGE1_MANIFEST_PATH,
    STAGE1_WINNER_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

query_contract = load_json(QUERY_CONTRACT_PATH)
retrieval_manifest = load_json(RETRIEVAL_ARTIFACT_MANIFEST_PATH)
stage1_manifest = load_json(STAGE1_MANIFEST_PATH)
stage1_winner = load_json(STAGE1_WINNER_MANIFEST_PATH)
stage1_selection = pd.read_csv(STAGE1_SELECTION_PATH)

if query_contract.get("active_query_column") != ACTIVE_QUERY_COLUMN:
    raise RuntimeError("Notebook 06 active query column must be query.")
if query_contract.get("evidence_scope") != "target_review_safe_signals_only":
    raise RuntimeError("Notebook 06 query evidence scope mismatch.")
if query_contract.get("user_prior_evidence_used") is not False:
    raise RuntimeError("Notebook 06 must not use user-prior evidence.")
if query_contract.get("historical_review_evidence_used") is not False:
    raise RuntimeError("Notebook 06 must not use historical-review evidence.")

if retrieval_manifest.get("evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 04 evidence scope mismatch.")
if retrieval_manifest.get("dense_source") != DENSE_TEXT_COLUMN:
    raise RuntimeError("Notebook 04 dense source must be dense_text.")
if retrieval_manifest.get("sparse_source") != SPARSE_TEXT_COLUMN:
    raise RuntimeError("Notebook 04 sparse source must be sparse_text.")
if retrieval_manifest.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Historical review-derived item signals must be enabled.")
if retrieval_manifest.get("review_reputation_graph_enabled") is not True:
    raise RuntimeError("Historical review-derived graph must be enabled.")
if retrieval_manifest.get("brand_in_functional_graph") is not False:
    raise RuntimeError("Brand must remain separate from product-functional facets.")
if retrieval_manifest.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 04 must preserve brand graph edges.")
if retrieval_manifest.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 04 must retain brand in production retrieval text.")
if retrieval_manifest.get("brand_in_profile_source_text") is not True:
    raise RuntimeError("Notebook 04 must retain brand in the profile-safe source.")
if retrieval_manifest.get("brand_in_synthetic_query") is not False:
    raise RuntimeError("Brand must remain excluded from the synthetic query.")

if stage1_manifest.get("evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 07 evidence scope mismatch.")
if stage1_manifest.get("query_evidence_scope") != "target_review_safe_signals_only":
    raise RuntimeError("Notebook 07 query evidence scope mismatch.")
if stage1_manifest.get("user_prior_enabled") is not False:
    raise RuntimeError("Notebook 07 baseline must not use user prior.")
if stage1_manifest.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Notebook 07 baseline must use historical review-derived item signals.")
if stage1_manifest.get("review_reputation_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 baseline must enable the historical review-derived graph.")
if stage1_manifest.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 must preserve brand as a catalog graph channel.")
if stage1_manifest.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 07 must retain brand in the item retrieval representation.")
if stage1_manifest.get("brand_in_candidate_output") is not True:
    raise RuntimeError("Notebook 07 candidate output must retain brand.")
if stage1_manifest.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 07 must not use synthetic-query brand matching.")
if stage1_manifest.get("candidate_pool_depth") != MAX_RETRIEVAL_K:
    raise RuntimeError("Notebook 07 candidate depth mismatch.")
if stage1_manifest.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 07 must use exact-K candidate budgets for all methods.")
if stage1_manifest.get("candidate_budget_k") != MAX_RETRIEVAL_K:
    raise RuntimeError("Notebook 07 exact-K budget mismatch.")
if stage1_manifest.get("exact_k_validation_passed") is not True:
    raise RuntimeError("Notebook 07 exact-K validation did not pass.")
if stage1_manifest.get("embedding_model") != EMBEDDING_MODEL_NAME:
    raise RuntimeError("Notebook 07 embedding model mismatch.")

if stage1_winner.get("contract_version") != "stage1_query_only_winner_v1":
    raise RuntimeError("Notebook 07 winner contract version mismatch.")
if stage1_winner.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 07 winner contract category mismatch.")
if stage1_winner.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 07 winner contract must use exact-K candidate budgets.")
if stage1_winner.get("candidate_budget_k") != MAX_RETRIEVAL_K:
    raise RuntimeError("Notebook 07 winner contract candidate budget mismatch.")
if stage1_winner.get("exact_k_validation_passed") is not True:
    raise RuntimeError("Notebook 07 winner contract did not pass exact-K validation.")
if stage1_winner.get("user_prior_enabled") is not False:
    raise RuntimeError("Notebook 07 winner must remain query-only.")
if stage1_winner.get("retrieval_evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 07 winner retrieval evidence scope mismatch.")
if stage1_winner.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 winner contract must preserve brand graph edges.")
if stage1_winner.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 07 winner contract must retain brand in item text.")
if stage1_winner.get("brand_in_candidate_output") is not True:
    raise RuntimeError("Notebook 07 winner contract must retain candidate brand.")
if stage1_winner.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 07 winner contract must keep query-side brand matching disabled.")

BASELINE_METHOD_SLUG = normalize_space(stage1_winner.get("winner_method_key"))
BASELINE_METHOD_LABEL = normalize_space(stage1_winner.get("winner_method_label"))
BASELINE_CANDIDATES_PATH = Path(normalize_space(stage1_winner.get("winner_candidate_path")))
if not BASELINE_METHOD_SLUG or not BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 07 winner contract has an empty method key or label.")
if BASELINE_METHOD_SLUG in PROFILE_METHODS:
    raise RuntimeError("The query-only winner method key conflicts with a personalized method key.")
if not BASELINE_CANDIDATES_PATH.exists():
    raise FileNotFoundError(f"Notebook 07 winner candidate cache is missing: {BASELINE_CANDIDATES_PATH}")

METHOD_ORDER = [BASELINE_METHOD_SLUG, *PROFILE_METHODS]
METHOD_LABELS = {BASELINE_METHOD_SLUG: BASELINE_METHOD_LABEL, **PROFILE_METHOD_LABELS}

require_columns(
    stage1_selection,
    ["selection_rank", "method_key", "retrieval_method", "is_selected_winner"],
    "Notebook 07 selection table",
)
selected_rows = stage1_selection.loc[boolean_series(stage1_selection["is_selected_winner"])].copy()
if len(selected_rows) != 1:
    raise RuntimeError("Notebook 07 selection table must contain exactly one selected winner.")
selected_row = selected_rows.iloc[0]
if normalize_space(selected_row["method_key"]) != BASELINE_METHOD_SLUG:
    raise RuntimeError("Notebook 07 selection table and winner contract method keys differ.")
if normalize_space(selected_row["retrieval_method"]) != BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 07 selection table and winner contract method labels differ.")
if stage1_manifest.get("winner_method_key") != BASELINE_METHOD_SLUG:
    raise RuntimeError("Notebook 07 run manifest and winner contract method keys differ.")
if stage1_manifest.get("winner_method_label") != BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 07 run manifest and winner contract method labels differ.")
manifest_candidate_path = stage1_manifest.get("candidate_paths", {}).get(BASELINE_METHOD_SLUG)
if not manifest_candidate_path or Path(manifest_candidate_path) != BASELINE_CANDIDATES_PATH:
    raise RuntimeError("Notebook 07 candidate path lineage does not match the winner contract.")

query_schema = pq.ParquetFile(QUERY_CACHE_PATH).schema.names
query_aliases = sorted({"query_C"}.intersection(query_schema))
expected_aliases = [] if COMPATIBILITY_QUERY_ALIAS is None else [COMPATIBILITY_QUERY_ALIAS]
if query_aliases != expected_aliases:
    raise RuntimeError(f"Active query aliases are ambiguous: expected {expected_aliases}, found {query_aliases}.")

required_query_columns = [
    "case_id",
    "user_id",
    "target_parent_asin",
    "regime",
    "target_timestamp_ms",
    ACTIVE_QUERY_COLUMN,
    "query_evidence_source",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "insufficient_review_evidence",
    "query_clean_is_active",
    *QUERY_PASSTHROUGH_COLUMNS,
]
if COMPATIBILITY_QUERY_ALIAS is not None:
    required_query_columns.append(COMPATIBILITY_QUERY_ALIAS)
queries = pd.read_parquet(QUERY_CACHE_PATH, columns=required_query_columns).copy()
require_columns(queries, required_query_columns, "query cache")

for column in ["case_id", "user_id", "target_parent_asin", "regime", ACTIVE_QUERY_COLUMN]:
    queries[column] = queries[column].fillna("").astype(str).map(normalize_space)
queries["target_timestamp_ms"] = pd.to_numeric(queries["target_timestamp_ms"], errors="raise").astype("int64")
queries["active_query_text"] = queries[ACTIVE_QUERY_COLUMN]

observed_query_regime_counts = queries["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict()
if len(set(observed_query_regime_counts.values())) != 1:
    raise RuntimeError(f"Query cache regime counts must be balanced: {observed_query_regime_counts}")
EXPECTED_REGIME_COUNTS = dict(observed_query_regime_counts)
EXPECTED_QUERY_ROWS = int(sum(EXPECTED_REGIME_COUNTS.values()))
if len(queries) != EXPECTED_QUERY_ROWS:
    raise RuntimeError(f"Expected {EXPECTED_QUERY_ROWS} query rows, found {len(queries)}.")
if queries["case_id"].duplicated().any():
    raise RuntimeError("case_id must be unique.")
if queries["user_id"].duplicated().any():
    raise RuntimeError("user_id must be unique.")
if queries[["case_id", "user_id", "target_parent_asin", "regime", "active_query_text"]].eq("").any().any():
    raise RuntimeError("Query cache contains an empty required value.")
if queries["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int).to_dict() != EXPECTED_REGIME_COUNTS:
    raise RuntimeError(f"Unexpected regime counts: {queries['regime'].value_counts().to_dict()}")
if not queries["query_evidence_source"].eq("target_review_safe_signals_only").all():
    raise RuntimeError("Every query must use target-review-safe signals only.")
for column in [
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "insufficient_review_evidence",
    "query_clean_is_active",
]:
    if boolean_series(queries[column]).any():
        raise RuntimeError(f"Query audit field must be false: {column}")
if COMPATIBILITY_QUERY_ALIAS is not None:
    if not queries[COMPATIBILITY_QUERY_ALIAS].fillna("").astype(str).map(normalize_space).eq(queries[ACTIVE_QUERY_COLUMN]).all():
        raise RuntimeError(f"{COMPATIBILITY_QUERY_ALIAS} must equal query.")

item_docs = pd.read_parquet(
    ITEM_DOCS_PATH,
    columns=["parent_asin", DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN],
).copy()
item_docs["parent_asin"] = item_docs["parent_asin"].fillna("").astype(str).map(normalize_space)
for column in [DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN]:
    item_docs[column] = item_docs[column].fillna("").astype(str).map(normalize_space)
if item_docs["parent_asin"].eq("").any() or item_docs["parent_asin"].duplicated().any():
    raise RuntimeError("Item documents contain empty or duplicate parent_asin values.")
if item_docs[DENSE_TEXT_COLUMN].eq("").any():
    raise RuntimeError("dense_text must be non-empty for every item.")
if item_docs[SPARSE_TEXT_COLUMN].eq("").any():
    raise RuntimeError("sparse_text must be non-empty for every item.")
if item_docs[BRAND_TEXT_COLUMN].eq("").all():
    raise RuntimeError("Notebook 04 item docs must retain non-empty brand facets.")
if len(item_docs) == 0:
    raise RuntimeError("Global Review catalog must contain at least one item.")

facet_columns = [
    "parent_asin",
    "facet_value_norm",
    "facet_role",
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
]
facets = pd.read_parquet(ITEM_FACETS_PATH, columns=facet_columns).copy()
require_columns(facets, facet_columns, "item facets")
for column in ["parent_asin", "facet_value_norm", "facet_role"]:
    facets[column] = facets[column].fillna("").astype(str).map(normalize_space)
for column in [
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
]:
    facets[column] = boolean_series(facets[column])

CORPORATE_SUFFIX_BRAND_FRAGMENTS = {
    "co",
    "company",
    "corp",
    "corporation",
    "gmbh",
    "inc",
    "incorporated",
    "limited",
    "llc",
    "ltd",
    "plc",
}

def is_suffix_only_brand_fragment(value):
    return normalize_entity_value(value) in CORPORATE_SUFFIX_BRAND_FRAGMENTS


all_brand_rows = facets.loc[facets["is_brand"]].copy()
brand_facet_rows = facets.loc[facets["is_brand_graph_facet"]].copy()
if all_brand_rows.empty:
    raise RuntimeError("Notebook 04 item facets must contain structured brand rows.")
if brand_facet_rows.empty:
    raise RuntimeError("Notebook 04 item facets must contain structured brand graph rows.")
if all_brand_rows["is_query_safe"].any():
    raise RuntimeError("Brand rows must remain query-unsafe.")
if all_brand_rows["is_product_functional_facet"].any():
    raise RuntimeError("Brand rows must remain separate from product-functional facets.")
if not brand_facet_rows["is_profile_safe"].all():
    raise RuntimeError("Brand graph facet rows must be profile-safe.")

brand_facet_rows["is_suffix_only_brand_fragment"] = brand_facet_rows["facet_value_norm"].map(
    is_suffix_only_brand_fragment
)
meaningful_brand_rows = brand_facet_rows.loc[~brand_facet_rows["is_suffix_only_brand_fragment"]].copy()
profile_safe_value_sets = (
    facets.loc[facets["is_profile_safe"] & facets["facet_value_norm"].ne("")]
    .groupby("parent_asin")["facet_value_norm"]
    .agg(lambda values: set(values))
    .to_dict()
)
meaningful_brand_value_sets = (
    meaningful_brand_rows.loc[meaningful_brand_rows["facet_value_norm"].ne("")]
    .groupby("parent_asin")["facet_value_norm"]
    .agg(lambda values: set(values))
    .to_dict()
)
suffix_only_fragment_count_by_item = (
    brand_facet_rows.loc[brand_facet_rows["is_suffix_only_brand_fragment"]]
    .groupby("parent_asin")
    .size()
    .astype(int)
    .to_dict()
)
brand_profile_validation_rows = []
for parent_asin, brand_values in meaningful_brand_value_sets.items():
    profile_values = profile_safe_value_sets.get(parent_asin, set())
    missing_values = sorted(brand_values - profile_values)
    brand_profile_validation_rows.append({
        "parent_asin": parent_asin,
        "meaningful_brand_segment_count": int(len(brand_values)),
        "matched_brand_segment_count": int(len(brand_values) - len(missing_values)),
        "missing_brand_segments": missing_values,
        "suffix_only_brand_fragment_count": int(suffix_only_fragment_count_by_item.get(parent_asin, 0)),
        "brand_profile_contract_pass": len(missing_values) == 0,
    })
brand_profile_qc = pd.DataFrame(brand_profile_validation_rows)
if brand_profile_qc.empty:
    meaningful_brand_item_count = 0
    brand_profile_violation_count = 0
else:
    meaningful_brand_item_count = int(len(brand_profile_qc))
    brand_profile_violation_count = int((~brand_profile_qc["brand_profile_contract_pass"]).sum())
suffix_only_fragment_count = int(brand_facet_rows["is_suffix_only_brand_fragment"].sum())
suffix_only_brand_items = sorted(
    set(brand_facet_rows.loc[brand_facet_rows["is_suffix_only_brand_fragment"], "parent_asin"])
    - set(meaningful_brand_value_sets.keys())
)
suffix_only_brand_item_count = int(len(suffix_only_brand_items))

if brand_profile_violation_count > 0:
    diagnostic_sample = brand_profile_qc.loc[
        ~brand_profile_qc["brand_profile_contract_pass"],
        ["parent_asin", "meaningful_brand_segment_count", "matched_brand_segment_count", "missing_brand_segments"],
    ].merge(
        item_docs[["parent_asin", BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN]],
        on="parent_asin",
        how="left",
    )[
        ["parent_asin", BRAND_TEXT_COLUMN, PROFILE_SAFE_TEXT_COLUMN, "missing_brand_segments"]
    ].head(20)
    display(diagnostic_sample)
    raise RuntimeError(
        "Meaningful brand facets are missing from the profile-safe facet "
        f"representation for {brand_profile_violation_count} items."
    )
print("Items with meaningful brand facets:", meaningful_brand_item_count)
print("Brand-profile violations:", brand_profile_violation_count)
print("Suffix-only brand fragments:", suffix_only_fragment_count)
print("Brand profile validation: passed")

item_docs = item_docs.sort_values("parent_asin", kind="stable").reset_index(drop=True)
catalog_size = int(len(item_docs))
EXPECTED_CANDIDATE_K = min(int(MAX_RETRIEVAL_K), catalog_size)
if EXPECTED_CANDIDATE_K <= 0:
    raise RuntimeError("Exact-K candidate budget must be positive.")
if stage1_manifest.get("catalog_size") != catalog_size:
    raise RuntimeError("Notebook 07 catalog size does not match the current Global Review catalog.")
if stage1_winner.get("effective_candidate_count_per_query") != EXPECTED_CANDIDATE_K:
    raise RuntimeError("Notebook 07 winner contract effective candidate count mismatch.")

prior_schema_columns = pq.ParquetFile(PRIOR_HISTORY_PATH).schema.names
forbidden_prior_fragments = (
    "review_text", "review_body", "review_title", "raw_review",
    "brand", "manufacturer", "seller", "rating", "sentiment",
    "prompt", "response", "llm",
)
forbidden_prior_columns = sorted(
    column for column in prior_schema_columns
    if any(fragment in column.lower() for fragment in forbidden_prior_fragments)
)
if forbidden_prior_columns:
    raise RuntimeError(f"Prior-history artifact contains forbidden fields: {forbidden_prior_columns}")

prior_columns = [
    "case_id",
    "user_id",
    "target_timestamp_ms",
    "prior_item_id",
    "prior_timestamp_ms",
]
if PRIOR_HAS_TARGET_PARENT_ASIN:
    prior_columns.append("target_parent_asin")
prior_history = pd.read_parquet(PRIOR_HISTORY_PATH, columns=prior_columns).copy()
require_columns(prior_history, prior_columns, "prior history")

baseline_schema = pq.ParquetFile(BASELINE_CANDIDATES_PATH).schema.names
required_baseline_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "candidate_parent_asin",
    "candidate_rank",
    "candidate_score",
    "method_key",
    "retrieval_method",
    "score_semantics",
    "candidate_brand_facet_text",
    "is_target",
]
missing_baseline_columns = [
    column for column in required_baseline_columns
    if column not in baseline_schema
]
if missing_baseline_columns:
    raise RuntimeError(f"Notebook 07 winner candidates are missing columns: {missing_baseline_columns}")
baseline_candidates = pd.read_parquet(
    BASELINE_CANDIDATES_PATH,
    columns=required_baseline_columns,
).copy()
require_columns(baseline_candidates, required_baseline_columns, "Notebook 07 winner candidates")
for column in [
    "case_id", "user_id", "regime", "target_parent_asin", "candidate_parent_asin",
    "method_key", "retrieval_method", "score_semantics", "candidate_brand_facet_text",
]:
    baseline_candidates[column] = baseline_candidates[column].fillna("").astype(str).map(normalize_space)
baseline_candidates["candidate_rank"] = pd.to_numeric(
    baseline_candidates["candidate_rank"], errors="raise"
).astype(int)
baseline_candidates["candidate_score"] = pd.to_numeric(
    baseline_candidates["candidate_score"], errors="raise"
).astype(float)
baseline_candidates["is_target"] = boolean_series(baseline_candidates["is_target"])
baseline_candidates["candidate_brand_facet_text"] = (
    baseline_candidates["candidate_brand_facet_text"].fillna("").astype(str).map(normalize_space)
)
item_brand_map = item_docs.set_index("parent_asin")[BRAND_TEXT_COLUMN]
expected_candidate_brand = (
    baseline_candidates["candidate_parent_asin"].map(item_brand_map).fillna("").astype(str)
)
if not baseline_candidates["candidate_brand_facet_text"].eq(expected_candidate_brand).all():
    raise RuntimeError("Notebook 07 candidate brand values do not match Notebook 04 item docs.")
if not baseline_candidates["method_key"].eq(BASELINE_METHOD_SLUG).all():
    raise RuntimeError("Notebook 07 winner candidate cache method_key mismatch.")
if not baseline_candidates["retrieval_method"].eq(BASELINE_METHOD_LABEL).all():
    raise RuntimeError("Notebook 07 winner candidate cache retrieval_method mismatch.")
baseline_candidates = baseline_candidates.sort_values(
    ["case_id", "candidate_rank"], kind="stable"
).reset_index(drop=True)

print("Rows: queries", len(queries))
print("Rows: items", len(item_docs))
print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Rows: baseline candidates", len(baseline_candidates))
print("Validation: input contracts passed")


Items with meaningful brand facets: 27151
Brand-profile violations: 0
Suffix-only brand fragments: 56
Brand profile validation: passed
Rows: queries 1968
Rows: items 27253
Query-only winner: hybrid_dense_bm25 - Dense-BM25 Hybrid
Rows: baseline candidates 1968000
Validation: input contracts passed


In [7]:
# ==== Validate Training-Safe History and Build Functional and Brand Profile Indexes ====
query_case_ids = set(queries["case_id"])
query_users = queries.set_index("case_id")["user_id"]
query_targets = queries.set_index("case_id")["target_parent_asin"]
query_timestamps = queries.set_index("case_id")["target_timestamp_ms"]
item_ids = item_docs["parent_asin"].tolist()
item_id_set = set(item_ids)


if set(baseline_candidates["case_id"]) != query_case_ids:
    raise RuntimeError("Notebook 07 winner candidate cases do not match the query cache.")
if baseline_candidates.duplicated(["case_id", "candidate_parent_asin"]).any():
    raise RuntimeError("Query-only winner candidate cache contains within-case duplicates.")
if not baseline_candidates["candidate_parent_asin"].isin(item_id_set).all():
    raise RuntimeError("Query-only winner candidate cache contains an item outside the Global Review catalog.")

baseline_counts = baseline_candidates.groupby("case_id").size()
if not baseline_counts.eq(EXPECTED_CANDIDATE_K).all():
    raise RuntimeError("Notebook 07 query-only winner baseline must provide exact-K candidates for every query.")
for case_id, group in baseline_candidates.groupby("case_id", sort=False):
    ranks = group["candidate_rank"].tolist()
    if ranks != list(range(1, EXPECTED_CANDIDATE_K + 1)):
        raise RuntimeError(f"Query-only winner candidate ranks are not exactly 1 through {EXPECTED_CANDIDATE_K} for case {case_id}.")
    scores = group["candidate_score"].to_numpy(dtype=float)
    if not np.isfinite(scores).all() or np.any(np.diff(scores) > 1e-12):
        raise RuntimeError(f"Query-only winner candidate scores are invalid for case {case_id}.")

prior_id_columns = ["case_id", "user_id", "prior_item_id"]
if PRIOR_HAS_TARGET_PARENT_ASIN:
    prior_id_columns.append("target_parent_asin")
for column in prior_id_columns:
    prior_history[column] = prior_history[column].fillna("").astype(str).map(normalize_space)
for column in ["target_timestamp_ms", "prior_timestamp_ms"]:
    prior_history[column] = pd.to_numeric(prior_history[column], errors="raise").astype("int64")

prior_history = prior_history[
    prior_history["case_id"].isin(query_case_ids)
    & prior_history["prior_item_id"].ne("")
].copy()
if len(prior_history):
    if not prior_history["user_id"].eq(prior_history["case_id"].map(query_users)).all():
        raise RuntimeError("Prior-history user_id does not match the query case.")
    if PRIOR_HAS_TARGET_PARENT_ASIN:
        if not prior_history["target_parent_asin"].eq(prior_history["case_id"].map(query_targets)).all():
            raise RuntimeError("Prior-history target item does not match the query case.")
    if not prior_history["target_timestamp_ms"].eq(prior_history["case_id"].map(query_timestamps)).all():
        raise RuntimeError("Prior-history target timestamp does not match the query case.")
    if not prior_history["prior_timestamp_ms"].lt(prior_history["target_timestamp_ms"]).all():
        raise RuntimeError("Prior history contains an interaction at or after the target timestamp.")
    if prior_history["prior_item_id"].eq(prior_history["case_id"].map(query_targets)).any():
        raise RuntimeError("Prior history contains the held-out target item.")

outside_item_universe = prior_history[~prior_history["prior_item_id"].isin(item_id_set)].copy()
outside_item_universe.to_csv(
    OUTPUT_DIR / f"prior_history_items_outside_item_universe_{CATEGORY_ID}.csv",
    index=False,
)
prior_history = prior_history[prior_history["prior_item_id"].isin(item_id_set)].copy()

prior_items = (
    prior_history
    .groupby(["case_id", "user_id", "prior_item_id"], as_index=False)
    .agg(
        prior_review_count=("prior_timestamp_ms", "size"),
        latest_prior_timestamp_ms=("prior_timestamp_ms", "max"),
        target_timestamp_ms=("target_timestamp_ms", "first"),
    )
    .sort_values(
        ["case_id", "latest_prior_timestamp_ms", "prior_item_id"],
        ascending=[True, False, True],
        kind="stable",
    )
    .reset_index(drop=True)
)

prior_event_counts = prior_history.groupby("case_id").size()
prior_item_counts = prior_items.groupby("case_id").size()
queries["prior_review_event_count"] = queries["case_id"].map(prior_event_counts).fillna(0).astype(int)
queries["prior_unique_item_count"] = queries["case_id"].map(prior_item_counts).fillna(0).astype(int)
queries["prior_depth_bin"] = queries["prior_unique_item_count"].map(prior_depth_bin)

for column in [
    "is_brand",
    "is_review_derived",
    "is_product_functional_facet",
    "is_query_safe",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
]:
    facets[column] = boolean_series(facets[column])
facets["parent_asin"] = facets["parent_asin"].fillna("").astype(str).map(normalize_space)
facets["facet_role"] = facets["facet_role"].fillna("").astype(str).map(normalize_space).str.lower()
facets["facet_value_norm"] = facets["facet_value_norm"].fillna("").astype(str).map(normalize_space).str.lower()

explicit_metadata_mask = (
    facets["is_product_functional_facet"]
    & facets["is_query_safe"]
    & ~facets["is_brand"]
    & ~facets["is_review_derived"]
    & ~facets["is_generic_category_anchor"]
    & ~facets["is_generic_utility_token"]
    & ~facets["is_context_dependent_utility_token"]
    & facets["is_metadata_facet_source"]
    & ~facets["is_disallowed_nonfacet_source"]
)
explicit_brand_mask = (
    facets["is_brand"]
    & ~facets["is_review_derived"]
    & facets["is_metadata_facet_source"]
    & ~facets["is_disallowed_nonfacet_source"]
)
explicit_profile_safe_mask = explicit_metadata_mask | explicit_brand_mask

if not facets["is_core_graph_facet"].eq(explicit_metadata_mask).all():
    raise RuntimeError("Notebook 04 is_core_graph_facet does not match the explicit metadata mask.")
if not facets["is_brand_graph_facet"].eq(explicit_brand_mask).all():
    raise RuntimeError("Notebook 04 is_brand_graph_facet does not match the explicit brand mask.")
if not facets["is_profile_safe"].eq(explicit_profile_safe_mask).all():
    raise RuntimeError("Notebook 04 is_profile_safe does not match functional-plus-brand profile policy.")

alignment_facets = facets.loc[
    explicit_metadata_mask
    & facets["parent_asin"].isin(item_id_set)
    & facets["facet_value_norm"].ne("")
].copy()
profile_safe_facets = facets.loc[
    explicit_profile_safe_mask
    & facets["parent_asin"].isin(item_id_set)
    & facets["facet_value_norm"].ne("")
].copy()

if alignment_facets.empty:
    raise RuntimeError("No metadata product-functional facets are available for QCHS prior-item alignment.")
if profile_safe_facets.empty:
    raise RuntimeError("No profile-safe functional or brand facets are available for QCHS expansion.")
if not alignment_facets["facet_role"].isin(FUNCTIONAL_ROLES).all():
    invalid_roles = sorted(
        set(alignment_facets.loc[~alignment_facets["facet_role"].isin(FUNCTIONAL_ROLES), "facet_role"])
    )
    raise RuntimeError(f"Unexpected QCHS alignment facet roles: {invalid_roles}")
if not profile_safe_facets["facet_role"].isin(PROFILE_ROLES).all():
    invalid_roles = sorted(
        set(profile_safe_facets.loc[~profile_safe_facets["facet_role"].isin(PROFILE_ROLES), "facet_role"])
    )
    raise RuntimeError(f"Unexpected profile-safe facet roles: {invalid_roles}")
if alignment_facets["is_brand"].any() or alignment_facets["is_review_derived"].any():
    raise RuntimeError("Brand or review-derived rows entered QCHS prior-item alignment.")
if profile_safe_facets["is_review_derived"].any():
    raise RuntimeError("Historical population-review facets must not enter the user profile.")
for column in [
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_disallowed_nonfacet_source",
]:
    if profile_safe_facets[column].any():
        raise RuntimeError(f"Disallowed rows entered the profile-safe facet channel: {column}")

brand_profile_facets = profile_safe_facets.loc[profile_safe_facets["is_brand"]].copy()
functional_profile_facets = profile_safe_facets.loc[~profile_safe_facets["is_brand"]].copy()
if brand_profile_facets.empty:
    raise RuntimeError("Brand facets must be available in the QCHS profile-safe channel.")
if brand_profile_facets["is_query_safe"].any():
    raise RuntimeError("Brand facets must remain query-unsafe.")
if brand_profile_facets["is_product_functional_facet"].any():
    raise RuntimeError("Brand facets must remain separate from product-functional facets.")
if not brand_profile_facets["is_profile_safe"].all():
    raise RuntimeError("Brand facets must be profile-safe.")

alignment_facets["facet_key"] = (
    alignment_facets["facet_role"] + "::" + alignment_facets["facet_value_norm"]
)
profile_safe_facets["facet_key"] = (
    profile_safe_facets["facet_role"] + "::" + profile_safe_facets["facet_value_norm"]
)
alignment_facets = alignment_facets.drop_duplicates(["parent_asin", "facet_key"]).copy()
profile_safe_facets = profile_safe_facets.drop_duplicates(["parent_asin", "facet_key"]).copy()

items_by_facet = (
    profile_safe_facets
    .sort_values(["facet_key", "parent_asin"], kind="stable")
    .groupby("facet_key", sort=False)["parent_asin"]
    .agg(list)
    .to_dict()
)

profile_facet_document_frequency = (
    profile_safe_facets.groupby("facet_key")["parent_asin"].nunique()
)
profile_facet_idf = {
    key: float(math.log((1.0 + len(item_docs)) / (1.0 + frequency)) + 1.0)
    for key, frequency in profile_facet_document_frequency.items()
}
alignment_facet_document_frequency = (
    alignment_facets.groupby("facet_key")["parent_asin"].nunique()
)
alignment_facet_idf = {
    key: float(math.log((1.0 + len(item_docs)) / (1.0 + frequency)) + 1.0)
    for key, frequency in alignment_facet_document_frequency.items()
}

alignment_item_token_sets = {}
alignment_records_by_item = defaultdict(list)
alignment_token_document_frequency = Counter()
for item_id, group in alignment_facets.groupby("parent_asin", sort=False):
    item_tokens = set()
    for row in group.itertuples(index=False):
        full_phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        phrase_tokens = tuple(
            token for token in full_phrase_tokens
            if token not in ATTENTION_STOPWORDS
        )
        if not full_phrase_tokens or not phrase_tokens:
            continue
        item_tokens.update(phrase_tokens)
        alignment_records_by_item[item_id].append({
            "facet_key": row.facet_key,
            "facet_role": row.facet_role,
            "phrase": row.facet_value_norm,
            "phrase_tokens": full_phrase_tokens,
            "tokens": phrase_tokens,
            "idf": alignment_facet_idf[row.facet_key],
            "is_brand": False,
        })
    alignment_item_token_sets[item_id] = item_tokens
    alignment_token_document_frequency.update(item_tokens)

alignment_token_idf = {
    token: float(math.log((1.0 + len(item_docs)) / (1.0 + frequency)) + 1.0)
    for token, frequency in alignment_token_document_frequency.items()
}

facet_records_by_item = defaultdict(list)
for item_id, group in profile_safe_facets.groupby("parent_asin", sort=False):
    for row in group.itertuples(index=False):
        full_phrase_tokens = tuple(canonical_tokens(row.facet_value_norm))
        phrase_tokens = tuple(
            token for token in full_phrase_tokens
            if token not in ATTENTION_STOPWORDS
        )
        if not full_phrase_tokens:
            continue
        facet_records_by_item[item_id].append({
            "facet_key": row.facet_key,
            "facet_role": row.facet_role,
            "phrase": row.facet_value_norm,
            "phrase_tokens": full_phrase_tokens,
            "tokens": phrase_tokens,
            "idf": profile_facet_idf[row.facet_key],
            "is_brand": bool(row.is_brand),
        })

prior_items_by_case = {
    case_id: group[["prior_item_id", "latest_prior_timestamp_ms", "prior_review_count"]].to_dict("records")
    for case_id, group in prior_items.groupby("case_id", sort=False)
}

qchs_alignable_item_ids = {
    item_id for item_id, records in alignment_records_by_item.items() if records
}
profile_safe_item_ids = {
    item_id for item_id, records in facet_records_by_item.items() if records
}
usable_stage1_profile_item_ids = qchs_alignable_item_ids & profile_safe_item_ids

usable_stage1_prior_items = prior_items[
    prior_items["prior_item_id"].isin(usable_stage1_profile_item_ids)
].copy()
profile_safe_prior_items = prior_items[
    prior_items["prior_item_id"].isin(profile_safe_item_ids)
].copy()
usable_stage1_prior_item_counts = usable_stage1_prior_items.groupby("case_id").size()
profile_safe_prior_item_counts = profile_safe_prior_items.groupby("case_id").size()
queries["stage1_usable_prior_item_count"] = (
    queries["case_id"].map(usable_stage1_prior_item_counts).fillna(0).astype(int)
)
queries["stage1_profile_safe_prior_item_count"] = (
    queries["case_id"].map(profile_safe_prior_item_counts).fillna(0).astype(int)
)
queries["stage1_profile_available"] = queries["stage1_usable_prior_item_count"].gt(0)

cold_history_mismatch = queries[
    queries["regime"].eq("cold")
    & (
        queries["prior_review_event_count"].ne(0)
        | queries["prior_unique_item_count"].ne(0)
        | queries["stage1_usable_prior_item_count"].ne(0)
        | queries["stage1_profile_safe_prior_item_count"].ne(0)
    )
].copy()
if len(cold_history_mismatch):
    display(cold_history_mismatch.head(20))
    raise RuntimeError("Cold cases must have zero usable pre-target Stage 1 history.")

non_cold_history_mismatch = queries[
    queries["regime"].ne("cold")
    & queries["prior_unique_item_count"].lt(1)
].copy()
if len(non_cold_history_mismatch):
    display(non_cold_history_mismatch.head(20))
    raise RuntimeError(
        "Every non-cold query must retain at least one training-safe pre-target prior item."
    )

non_cold_zero_qchs_alignable = queries[
    queries["regime"].ne("cold")
    & queries["stage1_usable_prior_item_count"].eq(0)
].copy()
non_cold_zero_profile_safe = queries[
    queries["regime"].ne("cold")
    & queries["stage1_profile_safe_prior_item_count"].eq(0)
].copy()

print("Rows: usable prior events", len(prior_history))
print("Rows: unique prior items", len(prior_items))
print("Rows: QCHS alignment facets", len(alignment_facets))
print("Rows: profile-safe functional facets", len(functional_profile_facets))
print("Rows: profile-safe brand facets", len(brand_profile_facets))
print("Non-cold cases with zero QCHS-alignable prior items:", len(non_cold_zero_qchs_alignable))
print("Non-cold cases with zero profile-safe prior items:", len(non_cold_zero_profile_safe))
print("Validation: prior history and brand-enabled profile facets passed")


Rows: usable prior events 9832
Rows: unique prior items 7848
Rows: QCHS alignment facets 102099
Rows: profile-safe functional facets 102099
Rows: profile-safe brand facets 27241
Non-cold cases with zero QCHS-alignable prior items: 6
Non-cold cases with zero profile-safe prior items: 0
Validation: prior history and brand-enabled profile facets passed


In [8]:
# ==== Define QCHS Alignment and Profile Construction ====
def prior_alignment(query_text, item_id):
    query_token_sequence = canonical_tokens(query_text)
    terms = stable_unique(
        token for token in query_token_sequence
        if token not in ATTENTION_STOPWORDS and len(token) > 1
    )
    records = alignment_records_by_item.get(item_id, [])
    if not terms or not records:
        return 0.0

    item_tokens = alignment_item_token_sets.get(item_id, set())
    token_score = weighted_overlap(terms, item_tokens, alignment_token_idf)
    exact_phrase_score = float(
        any(contains_token_sequence(query_token_sequence, record["phrase_tokens"]) for record in records)
    )
    matched_roles = {
        record["facet_role"]
        for record in records
        if set(record["tokens"]) & set(terms)
    }
    role_score = min(len(matched_roles) / 2.0, 1.0)
    return float(0.70 * token_score + 0.20 * exact_phrase_score + 0.10 * role_score)


def selected_facet_keys(selected_items):
    return stable_unique(
        record["facet_key"]
        for item_id in selected_items
        for record in facet_records_by_item.get(item_id, [])
    )


def build_anchors(selected_items, item_weights, query_text):
    # QCHS conditions prior-item selection on the query. Once an item is selected,
    # all of its profile-safe facets, including brand and query-confirming facets, are retained.
    _ = query_text
    anchor_scores = defaultdict(float)
    anchor_support = defaultdict(set)
    anchor_meta = {}

    for item_id, item_weight in zip(selected_items, item_weights):
        for record in facet_records_by_item.get(item_id, []):
            key = record["facet_key"]
            anchor_scores[key] += float(item_weight) * float(record["idf"])
            anchor_support[key].add(item_id)
            anchor_meta[key] = (
                record["facet_role"],
                record["phrase"],
                bool(record["is_brand"]),
            )

    ranked_keys = sorted(
        anchor_scores,
        key=lambda key: (
            -(anchor_scores[key] + 0.05 * math.log1p(len(anchor_support[key]))),
            anchor_meta[key][0],
            anchor_meta[key][1],
        ),
    )

    selected_keys = []
    selected_phrases = []
    selected_roles = []
    role_counts = Counter()
    for key in ranked_keys:
        role, phrase, _ = anchor_meta[key]
        if role_counts[role] >= MAX_ANCHOR_PHRASES_PER_ROLE:
            continue
        selected_keys.append(key)
        selected_phrases.append(phrase)
        selected_roles.append(role)
        role_counts[role] += 1
        if len(selected_keys) >= MAX_ANCHOR_PHRASES:
            break
    return selected_keys, selected_phrases, selected_roles


def empty_profile():
    return {
        "selected_items": [],
        "alignment_scores": [],
        "item_weights": [],
        "selected_facet_keys": [],
        "anchor_keys": [],
        "anchor_phrases": [],
        "anchor_roles": [],
        "attention_entropy": 0.0,
    }


def build_all_prior_profile(query_text, history_items):
    item_ids = [row["prior_item_id"] for row in history_items]
    if not item_ids:
        return empty_profile()
    weights = np.full(len(item_ids), 1.0 / len(item_ids), dtype=float)
    anchor_keys, anchor_phrases, anchor_roles = build_anchors(item_ids, weights, query_text)
    return {
        "selected_items": item_ids,
        "alignment_scores": [0.0] * len(item_ids),
        "item_weights": weights.tolist(),
        "selected_facet_keys": selected_facet_keys(item_ids),
        "anchor_keys": anchor_keys,
        "anchor_phrases": anchor_phrases,
        "anchor_roles": anchor_roles,
        "attention_entropy": normalized_entropy(weights),
    }


def build_qchs_profile(query_text, history_items):
    scored = []
    for row in history_items:
        score = prior_alignment(query_text, row["prior_item_id"])
        if score > 0:
            scored.append((score, int(row["latest_prior_timestamp_ms"]), row["prior_item_id"]))
    scored.sort(key=lambda value: (-value[0], -value[1], value[2]))
    scored = scored[:MAX_QCHS_PRIOR_ITEMS]

    item_ids = [item_id for _, _, item_id in scored]
    scores = [score for score, _, _ in scored]
    if not item_ids:
        return empty_profile()
    weights = softmax_weights(scores)
    anchor_keys, anchor_phrases, anchor_roles = build_anchors(item_ids, weights, query_text)
    return {
        "selected_items": item_ids,
        "alignment_scores": scores,
        "item_weights": weights.tolist(),
        "selected_facet_keys": selected_facet_keys(item_ids),
        "anchor_keys": anchor_keys,
        "anchor_phrases": anchor_phrases,
        "anchor_roles": anchor_roles,
        "attention_entropy": normalized_entropy(weights),
    }


In [9]:
# ==== Build Query-Conditioned Profiles and Fallback Flags ====
profile_by_case = {}
profile_rows = []

for row in queries.itertuples(index=False):
    history = prior_items_by_case.get(row.case_id, [])
    qchs_profile = build_qchs_profile(row.active_query_text, history)
    profile_by_case[row.case_id] = qchs_profile

    scores = qchs_profile["alignment_scores"]
    qchs_selected_prior_item_count = len(qchs_profile["selected_items"])
    qchs_profile_safe_facet_count = len(qchs_profile["selected_facet_keys"])
    qchs_brand_facet_count = sum(
        str(key).startswith("brand::") for key in qchs_profile["selected_facet_keys"]
    )
    qchs_functional_facet_count = sum(
        not str(key).startswith("brand::") for key in qchs_profile["selected_facet_keys"]
    )
    qchs_profile_available = bool(
        qchs_selected_prior_item_count > 0
        and qchs_profile_safe_facet_count > 0
    )
    profile_fallback_flag = not qchs_profile_available
    profile_fallback_reason = (
        "" if qchs_profile_available
        else "cold_no_history" if row.regime == "cold"
        else "no_qchs_aligned_prior"
    )

    profile_rows.append({
        "case_id": row.case_id,
        "user_id": row.user_id,
        "regime": row.regime,
        "target_parent_asin": row.target_parent_asin,
        "prior_review_event_count": int(row.prior_review_event_count),
        "prior_unique_item_count": int(row.prior_unique_item_count),
        "raw_prior_item_count": int(row.prior_unique_item_count),
        "training_safe_prior_item_count": int(row.prior_unique_item_count),
        "prior_depth_bin": row.prior_depth_bin,
        "stage1_usable_prior_item_count": int(row.stage1_usable_prior_item_count),
        "stage1_profile_safe_prior_item_count": int(row.stage1_profile_safe_prior_item_count),
        "stage1_profile_available": bool(row.stage1_profile_available),
        "qchs_selected_prior_item_count": qchs_selected_prior_item_count,
        "qchs_selected_ratio": (
            qchs_selected_prior_item_count / row.prior_unique_item_count
            if row.prior_unique_item_count else 0.0
        ),
        "qchs_alignment_mean": float(np.mean(scores)) if scores else 0.0,
        "qchs_alignment_max": float(np.max(scores)) if scores else 0.0,
        "qchs_attention_entropy": qchs_profile["attention_entropy"],
        "qchs_selected_facet_count": qchs_profile_safe_facet_count,
        "qchs_profile_safe_facet_count": qchs_profile_safe_facet_count,
        "qchs_selected_brand_facet_count": qchs_brand_facet_count,
        "qchs_brand_facet_count": qchs_brand_facet_count,
        "qchs_selected_functional_facet_count": qchs_functional_facet_count,
        "qchs_functional_facet_count": qchs_functional_facet_count,
        "qchs_anchor_phrase_count": len(qchs_profile["anchor_phrases"]),
        "qchs_anchor_brand_count": sum(
            role == "brand" for role in qchs_profile["anchor_roles"]
        ),
        "qchs_anchor_phrases": " | ".join(qchs_profile["anchor_phrases"]),
        "qchs_selected_prior_items": " | ".join(qchs_profile["selected_items"]),
        "qchs_profile_available": qchs_profile_available,
        "profile_fallback_flag": bool(profile_fallback_flag),
        "profile_fallback_reason": profile_fallback_reason,
        "qchs_fallback_flag": int(not qchs_profile["anchor_phrases"]),
    })

profile_diagnostics = pd.DataFrame(profile_rows)
profile_diagnostics["qchs_profile_available"] = profile_diagnostics["qchs_profile_available"].astype(bool)
profile_diagnostics["profile_fallback_flag"] = profile_diagnostics["profile_fallback_flag"].astype(bool)

cold_profile_mismatch = profile_diagnostics[
    profile_diagnostics["regime"].eq("cold")
    & (
        profile_diagnostics["qchs_selected_prior_item_count"].gt(0)
        | profile_diagnostics["qchs_selected_facet_count"].gt(0)
        | profile_diagnostics["qchs_anchor_phrase_count"].gt(0)
    )
].copy()
if len(cold_profile_mismatch):
    display(cold_profile_mismatch.head(20))
    raise RuntimeError("Cold cases must not produce a QCHS profile.")

non_cold_upstream_history_mismatch = profile_diagnostics[
    profile_diagnostics["regime"].ne("cold")
    & (
        profile_diagnostics["raw_prior_item_count"].lt(1)
        | profile_diagnostics["training_safe_prior_item_count"].lt(1)
    )
].copy()
if len(non_cold_upstream_history_mismatch):
    display(non_cold_upstream_history_mismatch.head(20))
    raise RuntimeError(
        "A non-cold case is missing valid strict/training-safe prior history. Fix upstream sampling."
    )

if (
    profile_diagnostics["qchs_profile_available"]
    & profile_diagnostics["profile_fallback_flag"]
).any():
    raise RuntimeError("A case cannot be both QCHS-profile-available and marked as profile fallback.")

fallback_counts_by_regime = (
    profile_diagnostics.loc[profile_diagnostics["profile_fallback_flag"], "regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
total_case_count = int(len(profile_diagnostics))
qchs_active_case_count = int(profile_diagnostics["qchs_profile_available"].sum())
fallback_case_count = int(profile_diagnostics["profile_fallback_flag"].sum())
non_cold_fallback_count = int(
    (
        profile_diagnostics["profile_fallback_flag"]
        & profile_diagnostics["regime"].ne("cold")
    ).sum()
)
fallback_rate = float(fallback_case_count / total_case_count) if total_case_count else 0.0

print("Rows: profile diagnostics", len(profile_diagnostics))
print("Rows: QCHS profiles with anchors", qchs_active_case_count)
print("Fallback counts by regime:", fallback_counts_by_regime)
print("Cases:", total_case_count)
print("QCHS-active cases:", qchs_active_case_count)
print("Fallback cases:", fallback_case_count)
print("Non-cold no-alignment fallbacks:", non_cold_fallback_count)
print("Fallback rate:", round(fallback_rate, 6))
print("Validation: QCHS profile availability and fallback diagnostics passed")


Rows: profile diagnostics 1968
Rows: QCHS profiles with anchors 553
Fallback counts by regime: {'cold': 656, 'weak': 487, 'strong': 272}
Cases: 1968
QCHS-active cases: 553
Fallback cases: 1415
Non-cold no-alignment fallbacks: 759
Fallback rate: 0.719004
Validation: QCHS profile availability and fallback diagnostics passed


In [10]:
# ==== Build the Personalized Retrieval Variants ====
if "baseline_candidates" not in globals():
    raise RuntimeError(
        "Run the complete 'Load Inputs and Validate Contracts' cell "
        "successfully before running personalized retrieval."
    )

sparse_index_start = time.perf_counter()
sparse_corpus_tokens = [tokenize_sparse_document(text) for text in item_docs[SPARSE_TEXT_COLUMN]]
if any(len(tokens) == 0 for tokens in sparse_corpus_tokens):
    raise RuntimeError("Every production sparse item document must contain an indexed token.")
bm25 = BM25Okapi(sparse_corpus_tokens)
offline_bm25_index_runtime_sec = time.perf_counter() - sparse_index_start

baseline_items_by_case = baseline_candidates.groupby("case_id", sort=False)["candidate_parent_asin"].agg(list).to_dict()
baseline_scores_by_case = baseline_candidates.groupby("case_id", sort=False)["candidate_score"].agg(list).to_dict()
query_lookup = queries.set_index("case_id")

sparse_qchs_items_by_case = {}
graph_qchs_items_by_case = {}
expanded_dense_text_by_case = {}

sparse_start = time.perf_counter()
for case_id in queries["case_id"]:
    profile = profile_by_case[case_id]
    if not profile["anchor_phrases"]:
        sparse_qchs_items_by_case[case_id] = []
        expanded_dense_text_by_case[case_id] = ""
        continue

    active_query_text = query_lookup.at[case_id, "active_query_text"]
    active_tokens = tokenize_sparse_query(active_query_text)
    anchor_tokens = [
        token
        for phrase in profile["anchor_phrases"]
        for token in tokenize_sparse_query(phrase)
    ]
    expanded_tokens = active_tokens * QUERY_REPEAT + anchor_tokens
    sparse_qchs_items_by_case[case_id] = top_exact_item_ids(
        bm25,
        expanded_tokens,
        item_ids,
        EXPECTED_CANDIDATE_K,
    )
    expanded_dense_text_by_case[case_id] = normalize_space(
        active_query_text + " " + " ".join(profile["anchor_phrases"])
    )
sparse_qchs_runtime_sec = time.perf_counter() - sparse_start

profile_case_ids = [
    case_id
    for case_id in queries["case_id"]
    if expanded_dense_text_by_case.get(case_id, "")
]
dense_qchs_items_by_case = {case_id: [] for case_id in queries["case_id"]}
offline_dense_index_runtime_sec = 0.0
dense_qchs_runtime_sec = 0.0

if profile_case_ids:
    dense_index_start = time.perf_counter()
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    item_embeddings = model.encode(
        item_docs[DENSE_TEXT_COLUMN].tolist(),
        batch_size=ITEM_EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype("float32")
    dense_index = faiss.IndexFlatIP(item_embeddings.shape[1])
    dense_index.add(item_embeddings)
    offline_dense_index_runtime_sec = time.perf_counter() - dense_index_start

    dense_query_start = time.perf_counter()
    dense_query_embeddings = model.encode(
        [expanded_dense_text_by_case[case_id] for case_id in profile_case_ids],
        batch_size=QUERY_EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype("float32")
    _, dense_positions = dense_index.search(dense_query_embeddings, EXPECTED_CANDIDATE_K)
    dense_qchs_runtime_sec = time.perf_counter() - dense_query_start

    for row_index, case_id in enumerate(profile_case_ids):
        dense_qchs_items_by_case[case_id] = [
            item_ids[int(position)]
            for position in dense_positions[row_index]
            if int(position) >= 0
        ]

graph_start = time.perf_counter()
for case_id in queries["case_id"]:
    profile = profile_by_case[case_id]
    graph_scores = defaultdict(float)
    for facet_key in profile["selected_facet_keys"]:
        for item_id in items_by_facet.get(facet_key, [])[:GRAPH_ITEMS_PER_FACET_LIMIT]:
            graph_scores[item_id] += 1.0
    graph_qchs_items_by_case[case_id] = top_items_from_score_map(
        graph_scores,
        EXPECTED_CANDIDATE_K,
    )
graph_qchs_runtime_sec = time.perf_counter() - graph_start

method_candidate_lists = {method: {} for method in METHOD_ORDER}
method_candidate_scores = {method: {} for method in METHOD_ORDER}
method_profile_source_used = {method: {} for method in PROFILE_METHODS}
fusion_runtime = defaultdict(float)

for case_id in queries["case_id"]:
    baseline_items = baseline_items_by_case[case_id]
    baseline_scores = baseline_scores_by_case[case_id]
    sparse_items = sparse_qchs_items_by_case.get(case_id, [])
    dense_items = dense_qchs_items_by_case.get(case_id, [])
    graph_items = graph_qchs_items_by_case.get(case_id, [])

    method_candidate_lists[BASELINE_METHOD_SLUG][case_id] = list(baseline_items)
    method_candidate_scores[BASELINE_METHOD_SLUG][case_id] = list(baseline_scores)

    method_sources = {
        "profile_sparse_qchs": ([baseline_items, sparse_items], PROFILE_SPARSE_WEIGHTS),
        "profile_hybrid_qchs": ([baseline_items, dense_items, sparse_items], PROFILE_HYBRID_WEIGHTS),
        "profile_full_qchs": ([baseline_items, dense_items, sparse_items, graph_items], PROFILE_FULL_WEIGHTS),
    }

    for method, (source_lists, weights) in method_sources.items():
        personalized_source_used = any(bool(source) for source in source_lists[1:])
        method_profile_source_used[method][case_id] = personalized_source_used
        if not personalized_source_used:
            method_candidate_lists[method][case_id] = list(baseline_items)
            method_candidate_scores[method][case_id] = list(baseline_scores)
            continue

        fusion_start = time.perf_counter()
        ranked, scores = rrf_fuse(source_lists, weights, MAX_RETRIEVAL_K)
        fusion_runtime[method] += time.perf_counter() - fusion_start
        method_candidate_lists[method][case_id] = ranked
        method_candidate_scores[method][case_id] = scores

method_runtime_components = {
    BASELINE_METHOD_SLUG: {
        "dense_qchs_runtime_sec": 0.0,
        "sparse_qchs_runtime_sec": 0.0,
        "graph_qchs_runtime_sec": 0.0,
        "fusion_runtime_sec": 0.0,
    },
    "profile_sparse_qchs": {
        "dense_qchs_runtime_sec": 0.0,
        "sparse_qchs_runtime_sec": float(sparse_qchs_runtime_sec),
        "graph_qchs_runtime_sec": 0.0,
        "fusion_runtime_sec": float(fusion_runtime["profile_sparse_qchs"]),
    },
    "profile_hybrid_qchs": {
        "dense_qchs_runtime_sec": float(dense_qchs_runtime_sec),
        "sparse_qchs_runtime_sec": float(sparse_qchs_runtime_sec),
        "graph_qchs_runtime_sec": 0.0,
        "fusion_runtime_sec": float(fusion_runtime["profile_hybrid_qchs"]),
    },
    "profile_full_qchs": {
        "dense_qchs_runtime_sec": float(dense_qchs_runtime_sec),
        "sparse_qchs_runtime_sec": float(sparse_qchs_runtime_sec),
        "graph_qchs_runtime_sec": float(graph_qchs_runtime_sec),
        "fusion_runtime_sec": float(fusion_runtime["profile_full_qchs"]),
    },
}

source_contribution = pd.DataFrame([
    {
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "baseline_query_only_winner_used": True,
        "dense_qchs_used": method in {"profile_hybrid_qchs", "profile_full_qchs"},
        "sparse_qchs_used": method in PROFILE_METHODS,
        "graph_qchs_used": method == "profile_full_qchs",
        "queries_with_personalized_source": (
            int(sum(method_profile_source_used[method].values()))
            if method in PROFILE_METHODS else 0
        ),
        "query_share_with_personalized_source": (
            float(np.mean(list(method_profile_source_used[method].values())))
            if method in PROFILE_METHODS else 0.0
        ),
        "mean_dense_qchs_candidate_count": (
            float(np.mean([len(dense_qchs_items_by_case[case_id]) for case_id in queries["case_id"]]))
            if method in {"profile_hybrid_qchs", "profile_full_qchs"} else 0.0
        ),
        "mean_sparse_qchs_candidate_count": (
            float(np.mean([len(sparse_qchs_items_by_case[case_id]) for case_id in queries["case_id"]]))
            if method in PROFILE_METHODS else 0.0
        ),
        "mean_graph_qchs_candidate_count": (
            float(np.mean([len(graph_qchs_items_by_case[case_id]) for case_id in queries["case_id"]]))
            if method == "profile_full_qchs" else 0.0
        ),
    }
    for method in METHOD_ORDER
])

print("Rows: Global Review item documents", len(item_docs))
print("Rows: queries with QCHS anchors", len(profile_case_ids))
print("Validation: four-method personalized retrieval passed")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/213 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Rows: Global Review item documents 27253
Rows: queries with QCHS anchors 553
Validation: four-method personalized retrieval passed


In [11]:
# ==== Enforce Exact Query-Only Fallback Identity ====
def _strict_cold_case_ids_for_stage1(queries_df, profile_lookup):
    required = {
        "case_id",
        "regime",
        "prior_review_event_count",
        "prior_unique_item_count",
        "stage1_usable_prior_item_count",
    }
    missing = sorted(required.difference(queries_df.columns))
    if missing:
        raise RuntimeError(f"Cold fallback source is missing required columns: {missing}")

    rows = []
    for row in queries_df.itertuples(index=False):
        case_id = str(row.case_id)
        regime = str(row.regime).strip().lower()

        prior_review_count = int(row.prior_review_event_count or 0)
        prior_unique_item_count = int(row.prior_unique_item_count or 0)
        stage1_usable_prior_item_count = int(row.stage1_usable_prior_item_count or 0)

        profile = profile_lookup.get(case_id, empty_profile())
        usable_profile_rows = int(
            len(profile.get("selected_items", []))
            + len(profile.get("selected_facet_keys", []))
            + len(profile.get("anchor_phrases", []))
        )

        actual_cold = (
            prior_review_count == 0
            and prior_unique_item_count == 0
            and stage1_usable_prior_item_count == 0
            and usable_profile_rows == 0
        )
        labeled_cold = regime == "cold"

        if labeled_cold != actual_cold:
            raise RuntimeError(
                "Cold regime label disagrees with strict pre-target history/profile evidence for "
                f"case_id={case_id}: regime={regime}, prior_review_count={prior_review_count}, "
                f"prior_unique_item_count={prior_unique_item_count}, "
                f"stage1_usable_prior_item_count={stage1_usable_prior_item_count}, "
                f"usable_profile_rows={usable_profile_rows}"
            )

        if actual_cold:
            rows.append(case_id)

    return set(rows)


def _assert_profile_method_cold_copy(method):
    checks = []
    for case_id in sorted(cold_case_ids):
        baseline_items = list(method_candidate_lists[BASELINE_METHOD_SLUG][case_id])
        method_items = list(method_candidate_lists[method][case_id])
        baseline_scores = np.asarray(method_candidate_scores[BASELINE_METHOD_SLUG][case_id], dtype=np.float64)
        method_scores = np.asarray(method_candidate_scores[method][case_id], dtype=np.float64)
        item_mismatch = baseline_items != method_items
        score_mismatch = not np.allclose(baseline_scores, method_scores, rtol=0.0, atol=1e-12)
        if item_mismatch or score_mismatch:
            raise RuntimeError(f"Cold candidate fallback identity failed for method={method}, case_id={case_id}")
        checks.append({
            "method_slug": method,
            "case_id": case_id,
            "candidate_count": len(method_items),
            "item_order_mismatch_count": int(item_mismatch),
            "score_mismatch_count": int(score_mismatch),
            "exact_identity_passed": True,
        })
    return checks


cold_case_ids = _strict_cold_case_ids_for_stage1(queries, profile_by_case)
for cold_case_id in cold_case_ids:
    baseline_items = list(method_candidate_lists[BASELINE_METHOD_SLUG][cold_case_id])
    baseline_scores = list(method_candidate_scores[BASELINE_METHOD_SLUG][cold_case_id])
    for profile_method in PROFILE_METHODS:
        method_candidate_lists[profile_method][cold_case_id] = list(baseline_items)
        method_candidate_scores[profile_method][cold_case_id] = list(baseline_scores)
        method_profile_source_used[profile_method][cold_case_id] = False

cold_candidate_fallback_qc = pd.DataFrame([
    row
    for profile_method in PROFILE_METHODS
    for row in _assert_profile_method_cold_copy(profile_method)
])

print("Cold candidate fallback policy: exact_query_only_winner_copy")
print("Cold queries copied from baseline:", len(cold_case_ids))


Cold candidate fallback policy: exact_query_only_winner_copy
Cold queries copied from baseline: 656


In [12]:
# ==== Evaluate the Query-Only and Personalized Methods ====
per_query_rows = []
candidate_list_rows = []
profile_diagnostics_by_case = profile_diagnostics.set_index("case_id")

score_source_by_method = {
    BASELINE_METHOD_SLUG: f"notebook07_{BASELINE_METHOD_SLUG}_score",
    "profile_sparse_qchs": "notebook08_query_only_winner_plus_sparse_qchs_rrf",
    "profile_hybrid_qchs": "notebook08_query_only_winner_plus_dense_sparse_qchs_rrf",
    "profile_full_qchs": "notebook08_query_only_winner_plus_dense_sparse_graph_qchs_rrf",
}

for method in METHOD_ORDER:
    for row in queries.itertuples(index=False):
        case_id = row.case_id
        ranked_items = method_candidate_lists[method][case_id]
        candidate_scores = method_candidate_scores[method][case_id]
        target_rank = ranked_items.index(row.target_parent_asin) + 1 if row.target_parent_asin in ranked_items else 0

        baseline_items = method_candidate_lists[BASELINE_METHOD_SLUG][case_id]
        baseline_rank = baseline_items.index(row.target_parent_asin) + 1 if row.target_parent_asin in baseline_items else 0
        baseline_set = set(baseline_items)
        method_set = set(ranked_items)

        metric_row = {
            "category_id": CATEGORY_ID,
            "case_id": case_id,
            "user_id": row.user_id,
            "regime": row.regime,
            "target_parent_asin": row.target_parent_asin,
            "method_slug": method,
            "method_label": METHOD_LABELS[method],
            "baseline_method_slug": BASELINE_METHOD_SLUG,
            "baseline_method_label": BASELINE_METHOD_LABEL,
            "rank": int(target_rank),
            "baseline_rank": int(baseline_rank),
            "target_newly_entered": int(target_rank > 0 and baseline_rank == 0),
            "target_newly_lost": int(target_rank == 0 and baseline_rank > 0),
            "both_hit_rank_improvement": (
                float(baseline_rank - target_rank)
                if baseline_rank > 0 and target_rank > 0
                else np.nan
            ),
            "candidate_jaccard_vs_baseline": (
                len(baseline_set & method_set) / len(baseline_set | method_set)
                if baseline_set or method_set else 1.0
            ),
        }
        for k in EVAL_KS:
            hit, ndcg, mrr = rank_metrics(target_rank, k)
            metric_row[f"HitRate@{k}"] = hit
            metric_row[f"NDCG@{k}"] = ndcg
            metric_row[f"MRR@{k}"] = mrr
        per_query_rows.append(metric_row)

        profile = profile_by_case.get(case_id, empty_profile())
        personalized_source_used = (
            method_profile_source_used[method][case_id]
            if method in PROFILE_METHODS else False
        )
        passthrough = {column: getattr(row, column) for column in QUERY_PASSTHROUGH_COLUMNS}
        candidate_list_rows.append({
            "category_id": CATEGORY_ID,
            "case_id": case_id,
            "query_id": case_id,
            "user_id": row.user_id,
            "regime": row.regime,
            **passthrough,
            "target_parent_asin": row.target_parent_asin,
            "target_timestamp_ms": int(row.target_timestamp_ms),
            "prior_history_n": int(row.prior_unique_item_count),
            "query": row.active_query_text,
            "baseline_method_slug": BASELINE_METHOD_SLUG,
            "baseline_method_label": BASELINE_METHOD_LABEL,
            "method_slug": method,
            "method_label": METHOD_LABELS[method],
            "candidate_pool_role": (
                "baseline_query_only" if method == BASELINE_METHOD_SLUG else "personalized_retrieval"
            ),
            "profile_fallback_flag": int(
                method != BASELINE_METHOD_SLUG and not personalized_source_used
            ),
            "profile_fallback_reason": (
                profile_diagnostics_by_case.at[case_id, "profile_fallback_reason"]
                if method != BASELINE_METHOD_SLUG and not personalized_source_used else ""
            ),
            "raw_prior_item_count": int(profile_diagnostics_by_case.at[case_id, "raw_prior_item_count"]),
            "training_safe_prior_item_count": int(profile_diagnostics_by_case.at[case_id, "training_safe_prior_item_count"]),
            "qchs_selected_prior_item_count": (
                int(profile_diagnostics_by_case.at[case_id, "qchs_selected_prior_item_count"])
                if method != BASELINE_METHOD_SLUG else 0
            ),
            "qchs_profile_safe_facet_count": (
                int(profile_diagnostics_by_case.at[case_id, "qchs_profile_safe_facet_count"])
                if method != BASELINE_METHOD_SLUG else 0
            ),
            "qchs_brand_facet_count": (
                int(profile_diagnostics_by_case.at[case_id, "qchs_brand_facet_count"])
                if method != BASELINE_METHOD_SLUG else 0
            ),
            "qchs_functional_facet_count": (
                int(profile_diagnostics_by_case.at[case_id, "qchs_functional_facet_count"])
                if method != BASELINE_METHOD_SLUG else 0
            ),
            "qchs_profile_available": (
                bool(profile_diagnostics_by_case.at[case_id, "qchs_profile_available"])
                if method != BASELINE_METHOD_SLUG else False
            ),
            "profile_selected_prior_item_count": (
                len(profile["selected_items"]) if method != BASELINE_METHOD_SLUG else 0
            ),
            "profile_selected_facet_count": (
                len(profile["selected_facet_keys"]) if method != BASELINE_METHOD_SLUG else 0
            ),
            "profile_anchor_phrase_count": (
                len(profile["anchor_phrases"]) if method != BASELINE_METHOD_SLUG else 0
            ),
            "profile_brand_facet_count": (
                sum(str(key).startswith("brand::") for key in profile["selected_facet_keys"])
                if method != BASELINE_METHOD_SLUG else 0
            ),
            "dense_qchs_candidate_count": (
                len(dense_qchs_items_by_case[case_id])
                if method in {"profile_hybrid_qchs", "profile_full_qchs"} else 0
            ),
            "sparse_qchs_candidate_count": (
                len(sparse_qchs_items_by_case[case_id])
                if method in PROFILE_METHODS else 0
            ),
            "graph_qchs_candidate_count": (
                len(graph_qchs_items_by_case[case_id])
                if method == "profile_full_qchs" else 0
            ),
            "candidate_parent_asin_list": ranked_items,
            "candidate_brand_facet_text_list": [
                item_brand_map.get(item_id, "") for item_id in ranked_items
            ],
            "candidate_score_list": candidate_scores,
            "candidate_count": len(ranked_items),
            "candidate_score_source": (
                score_source_by_method[method]
                if method == BASELINE_METHOD_SLUG or personalized_source_used
                else score_source_by_method[BASELINE_METHOD_SLUG]
            ),
        })

per_query_metrics = pd.DataFrame(per_query_rows)
candidate_lists = pd.DataFrame(candidate_list_rows)

results_by_pool_depth = pd.DataFrame([
    {
        "category_id": CATEGORY_ID,
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "candidate_pool_depth": k,
        "n_queries": int(len(group)),
        "HitRate": float(group[f"HitRate@{k}"].mean()),
        "NDCG": float(group[f"NDCG@{k}"].mean()),
        "MRR": float(group[f"MRR@{k}"].mean()),
    }
    for method, group in per_query_metrics.groupby("method_slug", sort=False)
    for k in EVAL_KS
])

results_by_pool_depth_regime = pd.DataFrame([
    {
        "category_id": CATEGORY_ID,
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "regime": regime,
        "candidate_pool_depth": k,
        "n_queries": int(len(group)),
        "HitRate": float(group[f"HitRate@{k}"].mean()),
        "NDCG": float(group[f"NDCG@{k}"].mean()),
        "MRR": float(group[f"MRR@{k}"].mean()),
    }
    for (method, regime), group in per_query_metrics.groupby(["method_slug", "regime"], sort=False)
    for k in EVAL_KS
])

results_overall = results_by_pool_depth[
    results_by_pool_depth["candidate_pool_depth"].eq(MAX_RETRIEVAL_K)
].reset_index(drop=True)
results_by_regime = results_by_pool_depth_regime[
    results_by_pool_depth_regime["candidate_pool_depth"].eq(MAX_RETRIEVAL_K)
].reset_index(drop=True)

movement_summary = pd.DataFrame([
    {
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        "regime": regime,
        "n_queries": int(len(group)),
        "target_newly_entered": int(group["target_newly_entered"].sum()),
        "target_newly_lost": int(group["target_newly_lost"].sum()),
        "both_hit_rank_improvement_mean": float(group["both_hit_rank_improvement"].mean()),
        "candidate_jaccard_vs_baseline_mean": float(group["candidate_jaccard_vs_baseline"].mean()),
    }
    for (method, regime), group in per_query_metrics.groupby(["method_slug", "regime"], sort=False)
])

print("Rows: per-query metrics", len(per_query_metrics))
print("Rows: candidate lists", len(candidate_lists))


Rows: per-query metrics 7872
Rows: candidate lists 7872


In [13]:
# ==== Summarize Stage-1 Exposure and Deep-Rank Movement ====
def _stage1_metric_at_rank(rank_value, k):
    rank_value = int(rank_value) if pd.notna(rank_value) else 0
    if rank_value <= 0 or rank_value > int(k):
        return 0.0, 0.0, 0.0
    return 1.0, float(1.0 / np.log2(rank_value + 1)), float(1.0 / rank_value)


if "PERSONALIZED_METHOD_SLUG" not in globals():
    _selection_metric = f"HitRate@{MAX_RETRIEVAL_K}"
    _selection_ndcg = f"NDCG@{MAX_RETRIEVAL_K}"
    _selection_mrr = f"MRR@{MAX_RETRIEVAL_K}"

    _profile_overall = (
        results_overall.loc[results_overall["method_slug"].isin(PROFILE_METHODS)]
        .rename(columns={"HitRate": _selection_metric, "NDCG": _selection_ndcg, "MRR": _selection_mrr})
        .copy()
    )

    _regime_consistency = (
        results_by_regime.loc[results_by_regime["method_slug"].isin(PROFILE_METHODS)]
        .groupby(["method_slug", "method_label"], observed=True)["HitRate"]
        .agg(
            regime_hit_rate_min="min",
            regime_hit_rate_std=lambda values: float(values.std(ddof=0)),
        )
        .reset_index()
    )

    _method_runtime = pd.DataFrame([
        {
            "method_slug": method,
            "online_personalized_runtime_sec": float(sum(method_runtime_components[method].values())),
        }
        for method in METHOD_ORDER
    ])

    _selection_table = (
        _profile_overall
        .merge(_regime_consistency, on=["method_slug", "method_label"], how="left", validate="one_to_one")
        .merge(_method_runtime, on="method_slug", how="left", validate="one_to_one")
    )
    _selection_table["method_order_index"] = _selection_table["method_slug"].map(
        {method: index for index, method in enumerate(PROFILE_METHODS)}
    )

    _selection_table = _selection_table.sort_values(
        [
            _selection_metric,
            _selection_ndcg,
            _selection_mrr,
            "regime_hit_rate_min",
            "regime_hit_rate_std",
            "online_personalized_runtime_sec",
            "method_order_index",
        ],
        ascending=[False, False, False, False, True, True, True],
        kind="stable",
    ).reset_index(drop=True)

    PERSONALIZED_METHOD_SLUG = str(_selection_table.iloc[0]["method_slug"])


def _stage1_uplift_frame(group_cols):
    selected = per_query_metrics.loc[
        per_query_metrics["method_slug"].isin([BASELINE_METHOD_SLUG, PERSONALIZED_METHOD_SLUG])
    ].copy()

    metrics = ["HitRate@1000", "MRR@1000", "NDCG@1000", "NDCG@5", "NDCG@1"]

    if group_cols:
        grouped = (
            selected.groupby(group_cols + ["method_slug"], dropna=False, sort=False)[metrics]
            .mean()
            .reset_index()
        )
        base = grouped.loc[grouped["method_slug"].eq(BASELINE_METHOD_SLUG)].drop(columns=["method_slug"])
        pers = grouped.loc[grouped["method_slug"].eq(PERSONALIZED_METHOD_SLUG)].drop(columns=["method_slug"])
        out = base.merge(pers, on=group_cols, suffixes=("_baseline", "_personalized"), validate="one_to_one")
    else:
        grouped = selected.groupby(["method_slug"], dropna=False, sort=False)[metrics].mean().reset_index()
        base = grouped.loc[grouped["method_slug"].eq(BASELINE_METHOD_SLUG), metrics].reset_index(drop=True)
        pers = grouped.loc[grouped["method_slug"].eq(PERSONALIZED_METHOD_SLUG), metrics].reset_index(drop=True)
        out = pd.concat([base.add_suffix("_baseline"), pers.add_suffix("_personalized")], axis=1)

    for metric in metrics:
        out[f"delta_{metric}"] = out[f"{metric}_personalized"] - out[f"{metric}_baseline"]

    return out


stage1_personalization_uplift_overall = _stage1_uplift_frame([])
stage1_personalization_uplift_by_regime = _stage1_uplift_frame(["regime"])

_cold_uplift = stage1_personalization_uplift_by_regime.loc[
    stage1_personalization_uplift_by_regime["regime"].astype(str).str.lower().eq("cold")
]
_delta_cols = [col for col in stage1_personalization_uplift_by_regime.columns if col.startswith("delta_")]

if len(_cold_uplift) != 1 or not np.allclose(
    _cold_uplift[_delta_cols].to_numpy(dtype=float),
    0.0,
    rtol=0.0,
    atol=1e-12,
):
    raise RuntimeError("Cold Stage 1 personalization uplift must be exactly zero for every metric.")

stage1_personalization_uplift_overall.to_csv(
    OUTPUT_DIR / f"stage1_personalization_uplift_overall_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)

stage1_personalization_uplift_by_regime.to_csv(
    OUTPUT_DIR / f"stage1_personalization_uplift_by_regime_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)

In [14]:
# ==== Compare Personalized Variants and Lock the Winner ====
contrast_specs = {
    f"{method}_minus_query_only_winner": (method, BASELINE_METHOD_SLUG)
    for method in PROFILE_METHODS
}
metric_columns = [f"{metric}@{MAX_RETRIEVAL_K}" for metric in ["HitRate", "NDCG", "MRR"]]
wide = per_query_metrics.pivot(
    index=["case_id", "user_id", "regime"],
    columns="method_slug",
    values=metric_columns,
)

contrast_rows = []
for contrast_name, (left_method, right_method) in contrast_specs.items():
    for case_key, row in wide.iterrows():
        output = {
            "case_id": case_key[0],
            "user_id": case_key[1],
            "regime": case_key[2],
            "contrast": contrast_name,
            "left_method": left_method,
            "right_method": right_method,
        }
        for metric in metric_columns:
            output[f"delta_{metric}"] = float(row[(metric, left_method)] - row[(metric, right_method)])
        contrast_rows.append(output)
paired_contrasts = pd.DataFrame(contrast_rows)

contrast_summary_rows = []
for contrast_name, group in paired_contrasts.groupby("contrast", sort=False):
    grouped_frames = [("overall", group), *list(group.groupby("regime", sort=False))]
    for regime, regime_group in grouped_frames:
        output = {
            "contrast": contrast_name,
            "regime": regime,
            "n_queries": int(len(regime_group)),
        }
        for metric in metric_columns:
            output[f"mean_delta_{metric}"] = float(regime_group[f"delta_{metric}"].mean())
        contrast_summary_rows.append(output)
contrast_summary = pd.DataFrame(contrast_summary_rows)

runtime_summary = pd.DataFrame([
    {
        "method_slug": method,
        "method_label": METHOD_LABELS[method],
        **method_runtime_components[method],
        "online_personalized_runtime_sec": float(sum(method_runtime_components[method].values())),
        "offline_profile_sparse_index_runtime_sec": float(offline_bm25_index_runtime_sec),
        "offline_dense_index_runtime_sec": float(offline_dense_index_runtime_sec),
        "baseline_source_runtime_excluded": True,
    }
    for method in METHOD_ORDER
])

selection_metric = f"HitRate@{MAX_RETRIEVAL_K}"
selection_ndcg = f"NDCG@{MAX_RETRIEVAL_K}"
selection_mrr = f"MRR@{MAX_RETRIEVAL_K}"
profile_overall = (
    results_overall.loc[results_overall["method_slug"].isin(PROFILE_METHODS)]
    .rename(columns={"HitRate": selection_metric, "NDCG": selection_ndcg, "MRR": selection_mrr})
    .copy()
)
regime_consistency = (
    results_by_regime.loc[results_by_regime["method_slug"].isin(PROFILE_METHODS)]
    .groupby(["method_slug", "method_label"], observed=True)["HitRate"]
    .agg(
        regime_hit_rate_min="min",
        regime_hit_rate_max="max",
        regime_hit_rate_mean="mean",
        regime_hit_rate_std=lambda values: float(values.std(ddof=0)),
    )
    .reset_index()
)
personalized_method_selection = (
    profile_overall
    .merge(regime_consistency, on=["method_slug", "method_label"], how="left", validate="one_to_one")
    .merge(
        runtime_summary[["method_slug", "online_personalized_runtime_sec"]],
        on="method_slug",
        how="left",
        validate="one_to_one",
    )
)
personalized_method_selection["method_order_index"] = personalized_method_selection["method_slug"].map(
    {method: index for index, method in enumerate(PROFILE_METHODS)}
)
personalized_method_selection = personalized_method_selection.sort_values(
    [
        selection_metric,
        selection_ndcg,
        selection_mrr,
        "regime_hit_rate_min",
        "regime_hit_rate_std",
        "online_personalized_runtime_sec",
        "method_order_index",
    ],
    ascending=[False, False, False, False, True, True, True],
    kind="stable",
).reset_index(drop=True)
personalized_method_selection.insert(
    0,
    "selection_rank",
    np.arange(1, len(personalized_method_selection) + 1),
)
personalized_method_selection["primary_stage1_metric"] = selection_metric
personalized_method_selection["selection_rule"] = (
    f"maximize {selection_metric}; then {selection_ndcg}, {selection_mrr}, minimum regime HitRate, "
    "regime HitRate stability, and online personalized runtime"
)

automatic_personalized_winner = str(personalized_method_selection.iloc[0]["method_slug"])
if PERSONALIZED_WINNER_METHOD_OVERRIDE is None:
    PERSONALIZED_METHOD_SLUG = automatic_personalized_winner
    PERSONALIZED_SELECTION_SOURCE = "automatic_selection_rank_1"
else:
    PERSONALIZED_METHOD_SLUG = str(PERSONALIZED_WINNER_METHOD_OVERRIDE).strip()
    if PERSONALIZED_METHOD_SLUG not in PROFILE_METHODS:
        raise RuntimeError(
            f"PERSONALIZED_WINNER_METHOD_OVERRIDE must be one of {PROFILE_METHODS}, "
            f"found {PERSONALIZED_METHOD_SLUG!r}."
        )
    PERSONALIZED_SELECTION_SOURCE = "config_override"
PERSONALIZED_METHOD_LABEL = PROFILE_METHOD_LABELS[PERSONALIZED_METHOD_SLUG]

personalized_method_selection["is_selected_winner"] = personalized_method_selection["method_slug"].eq(
    PERSONALIZED_METHOD_SLUG
)
personalized_method_selection["selection_status"] = np.where(
    personalized_method_selection["is_selected_winner"],
    "selected_for_downstream",
    "not_selected",
)
personalized_method_selection["winner_selection_source"] = PERSONALIZED_SELECTION_SOURCE
personalized_method_selection = personalized_method_selection.drop(columns=["method_order_index"])
personalized_winner_row = personalized_method_selection.loc[
    personalized_method_selection["is_selected_winner"]
].iloc[0]

candidate_lists["selected_personalized_method"] = PERSONALIZED_METHOD_SLUG
candidate_lists["selected_personalized_method_label"] = PERSONALIZED_METHOD_LABEL
candidate_lists["is_selected_personalized_method"] = candidate_lists["method_slug"].eq(
    PERSONALIZED_METHOD_SLUG
)

print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Selected personalized winner:", PERSONALIZED_METHOD_SLUG, "-", PERSONALIZED_METHOD_LABEL)


Query-only winner: hybrid_dense_bm25 - Dense-BM25 Hybrid
Selected personalized winner: profile_sparse_qchs - Profile Sparse QCHS


In [15]:
# ==== Validate and Export Personalized Retrieval Artifacts ====
required_runtime_frames = [
    "candidate_lists",
    "per_query_metrics",
    "results_overall",
    "results_by_pool_depth",
    "results_by_regime",
    "personalized_method_selection",
]

missing_runtime_frames = [
    name for name in required_runtime_frames
    if name not in globals()
]

if missing_runtime_frames:
    raise RuntimeError(
        "Run the candidate-list and metrics construction cells before Validation and Export. "
        f"Missing: {missing_runtime_frames}"
    )

expected_methods = [BASELINE_METHOD_SLUG, *PROFILE_METHODS]
if METHOD_ORDER != expected_methods:
    raise RuntimeError(f"Notebook 08 methods must be exactly {expected_methods}.")
if set(per_query_metrics["method_slug"]) != set(expected_methods):
    raise RuntimeError("Per-query metrics do not contain the exact baseline and profile methods.")

expected_rows = len(queries) * len(METHOD_ORDER)
if len(candidate_lists) != expected_rows or len(per_query_metrics) != expected_rows:
    raise RuntimeError("Output row count must equal queries multiplied by the evaluated methods.")
if candidate_lists.duplicated(["case_id", "method_slug"]).any():
    raise RuntimeError("Candidate lists contain duplicate case-method rows.")
if not candidate_lists["candidate_count"].eq(EXPECTED_CANDIDATE_K).all():
    bad_counts = (
        candidate_lists.loc[
            ~candidate_lists["candidate_count"].eq(EXPECTED_CANDIDATE_K),
            ["case_id", "method_slug", "candidate_count"],
        ]
        .head(10)
        .to_dict("records")
    )
    raise RuntimeError(f"Candidate counts must equal exact-K={EXPECTED_CANDIDATE_K}: {bad_counts}")

expected_case_ids = set(queries["case_id"].astype(str))
for method, group in candidate_lists.groupby("method_slug", sort=False):
    if set(group["case_id"].astype(str)) != expected_case_ids:
        raise RuntimeError(f"{method} does not cover the same query IDs as the query cache.")

for row in candidate_lists.itertuples(index=False):
    items = list(row.candidate_parent_asin_list)
    brands = list(row.candidate_brand_facet_text_list)
    scores = np.asarray(row.candidate_score_list, dtype=float)
    if len(items) != row.candidate_count or len(scores) != row.candidate_count:
        raise RuntimeError("Candidate item and score lengths do not match candidate_count.")
    if len(brands) != row.candidate_count:
        raise RuntimeError("Candidate brand list length does not match candidate_count.")
    expected_brands = [item_brand_map.get(item_id, "") for item_id in items]
    if brands != expected_brands:
        raise RuntimeError("Candidate brand values do not match Notebook 04 item docs.")
    if len(items) != EXPECTED_CANDIDATE_K:
        raise RuntimeError("Candidate list order must define ranks 1 through exact-K.")
    if len(set(items)) != len(items):
        raise RuntimeError("A candidate list contains duplicate items.")
    if not set(items).issubset(item_id_set):
        raise RuntimeError("A candidate list contains an item outside the Global Review catalog.")
    if not np.isfinite(scores).all() or np.any(np.diff(scores) > 1e-12):
        raise RuntimeError("Candidate scores must be finite and non-increasing.")

for case_id in queries["case_id"]:
    if method_candidate_lists[BASELINE_METHOD_SLUG][case_id] != baseline_items_by_case[case_id]:
        raise RuntimeError("The query-only winner order changed from Notebook 07.")
    if method_candidate_scores[BASELINE_METHOD_SLUG][case_id] != baseline_scores_by_case[case_id]:
        raise RuntimeError("The query-only winner scores changed from Notebook 07.")
    for method in PROFILE_METHODS:
        if not method_profile_source_used[method][case_id]:
            if method_candidate_lists[method][case_id] != baseline_items_by_case[case_id]:
                raise RuntimeError("A profile fallback did not preserve the query-only winner order.")
            if method_candidate_scores[method][case_id] != baseline_scores_by_case[case_id]:
                raise RuntimeError("A profile fallback did not preserve the query-only winner scores.")

cold_case_ids = queries.loc[queries["regime"].eq("cold"), "case_id"]
for case_id in cold_case_ids:
    for method in PROFILE_METHODS:
        if method_candidate_lists[method][case_id] != baseline_items_by_case[case_id]:
            raise RuntimeError("A cold case did not preserve the query-only winner order.")
        if method_candidate_scores[method][case_id] != baseline_scores_by_case[case_id]:
            raise RuntimeError("A cold case did not preserve the query-only winner scores.")

non_cold_case_ids = queries.loc[queries["regime"].ne("cold"), "case_id"]
profile_diag_by_case = profile_diagnostics.set_index("case_id")
fallback_case_ids = set(
    profile_diagnostics.loc[profile_diagnostics["profile_fallback_flag"], "case_id"].astype(str)
)
qchs_active_case_ids = set(
    profile_diagnostics.loc[profile_diagnostics["qchs_profile_available"], "case_id"].astype(str)
)
if fallback_case_ids & qchs_active_case_ids:
    raise RuntimeError("A case cannot be both fallback and QCHS-active.")
for case_id in queries["case_id"]:
    expected_fallback = case_id in fallback_case_ids
    for method in PROFILE_METHODS:
        if not method_profile_source_used[method][case_id]:
            if not expected_fallback:
                raise RuntimeError(f"Non-fallback case used baseline fallback: method={method}, case_id={case_id}")
            if method_candidate_lists[method][case_id] != baseline_items_by_case[case_id]:
                raise RuntimeError("Fallback candidate IDs/order must equal the baseline query-only list.")
            if method_candidate_scores[method][case_id] != baseline_scores_by_case[case_id]:
                raise RuntimeError("Fallback candidate scores must equal the baseline query-only scores.")
        elif expected_fallback:
            raise RuntimeError(f"Fallback case used an active personalized source: method={method}, case_id={case_id}")

if queries.loc[queries["regime"].ne("cold"), "prior_unique_item_count"].lt(1).any():
    raise RuntimeError("Non-cold queries must have at least one training-safe prior item.")
if queries.loc[queries["regime"].eq("cold"), "stage1_usable_prior_item_count"].ne(0).any():
    raise RuntimeError("Cold queries must have zero usable Stage 1 prior items.")

if alignment_facets["is_brand"].any() or alignment_facets["is_review_derived"].any():
    raise RuntimeError("Brand or review-derived rows entered QCHS prior-item alignment.")
if profile_safe_facets["is_review_derived"].any():
    raise RuntimeError("Historical population-review facets entered the user-profile channel.")
if brand_profile_facets.empty:
    raise RuntimeError("Brand facets must be present in the personalized profile channel.")
if brand_profile_facets["is_query_safe"].any():
    raise RuntimeError("Brand profile facets must remain query-unsafe.")
if brand_profile_facets["is_product_functional_facet"].any():
    raise RuntimeError("Brand profile facets must remain separate from functional facets.")
if not brand_profile_facets["is_profile_safe"].all():
    raise RuntimeError("Brand profile facets must be profile-safe.")
if profile_safe_facets[
    [
        "is_generic_category_anchor",
        "is_generic_utility_token",
        "is_context_dependent_utility_token",
        "is_disallowed_nonfacet_source",
    ]
].any().any():
    raise RuntimeError("Generic or disallowed rows entered the profile-safe facet channel.")

if int(personalized_method_selection["is_selected_winner"].sum()) != 1:
    raise RuntimeError("Personalized method-selection table must contain exactly one selected winner.")
selected_slug = str(
    personalized_method_selection.loc[
        personalized_method_selection["is_selected_winner"], "method_slug"
    ].iloc[0]
)
if selected_slug != PERSONALIZED_METHOD_SLUG:
    raise RuntimeError("Personalized winner selection variables and table differ.")
if PERSONALIZED_METHOD_SLUG not in PROFILE_METHODS:
    raise RuntimeError("The selected personalized winner must be a personalized retrieval method.")
selected_candidate_rows = candidate_lists.loc[
    boolean_series(candidate_lists["is_selected_personalized_method"])
].copy()
if len(selected_candidate_rows) != len(queries):
    raise RuntimeError("The selected personalized method must have exactly one flagged row per query.")
if set(selected_candidate_rows["case_id"].astype(str)) != expected_case_ids:
    raise RuntimeError("Selected-personalized flags do not cover the full query set.")
if not selected_candidate_rows["method_slug"].eq(PERSONALIZED_METHOD_SLUG).all():
    raise RuntimeError("Selected-personalized flags identify a non-winner method.")
if not candidate_lists["selected_personalized_method_label"].eq(PERSONALIZED_METHOD_LABEL).all():
    raise RuntimeError("Candidate lists contain an inconsistent personalized winner label.")

forbidden_output_fragments = (
    "review_text", "review_body", "review_title", "raw_review",
    "rating", "sentiment", "prompt", "response", "llm",
)
for frame_name, frame in [
    ("candidate_lists", candidate_lists),
    ("per_query_metrics", per_query_metrics),
    ("profile_diagnostics", profile_diagnostics),
]:
    forbidden = sorted(
        column for column in frame.columns
        if any(fragment in column.lower() for fragment in forbidden_output_fragments)
    )
    if forbidden:
        raise RuntimeError(f"{frame_name} contains forbidden prior-evidence fields: {forbidden}")

print("Cases:", total_case_count)
print("QCHS-active cases:", qchs_active_case_count)
print("Fallback cases:", fallback_case_count)
print("Non-cold no-alignment fallbacks:", non_cold_fallback_count)
print("Fallback rate:", round(fallback_rate, 6))
print("Fallback counts by regime:", fallback_counts_by_regime)
print("Candidate fallback validation: passed")

candidate_lists.to_parquet(CANDIDATE_LISTS_PATH, index=False)
per_query_metrics.to_parquet(PER_QUERY_METRICS_PATH, index=False)
profile_diagnostics.to_csv(PROFILE_DIAGNOSTICS_PATH, index=False, encoding="utf-8-sig")
personalized_method_selection.to_csv(METHOD_SELECTION_PATH, index=False, encoding="utf-8-sig")

results_overall.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_overall_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
results_by_pool_depth.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_by_pool_depth_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
results_by_regime.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_by_regime_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
results_by_pool_depth_regime.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_results_by_pool_depth_regime_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
source_contribution.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_source_contribution_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
movement_summary.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_candidate_movement_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
paired_contrasts.to_parquet(
    OUTPUT_DIR / f"personalized_retrieval_paired_contrasts_{CATEGORY_ID}.parquet",
    index=False,
)
contrast_summary.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_contrast_summary_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)
runtime_summary.to_csv(
    OUTPUT_DIR / f"personalized_retrieval_runtime_top1000_{CATEGORY_ID}.csv",
    index=False,
    encoding="utf-8-sig",
)

run_manifest = {
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "query_rows": int(len(queries)),
    "expected_regime_counts": EXPECTED_REGIME_COUNTS,
    "candidate_pool_depth": MAX_RETRIEVAL_K,
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": MAX_RETRIEVAL_K,
    "expected_candidate_count_per_query": int(EXPECTED_CANDIDATE_K),
    "catalog_size": int(catalog_size),
    "exact_k_validation_passed": True,
    "cold_retrieval_fallback_policy": "exact_query_only_winner_copy",
    "cold_retrieval_identity_passed": True,
    "n_cold_queries": int(len(cold_case_ids)),
    "regime_source": "strict_pre_target_history",
    "qchs_profile_coverage_rate": float(qchs_active_case_count / total_case_count) if total_case_count else 0.0,
    "qchs_active_case_count": int(qchs_active_case_count),
    "fallback_case_count": int(fallback_case_count),
    "non_cold_fallback_case_count": int(non_cold_fallback_count),
    "fallback_policy": "copy_baseline_candidates",
    "fallback_changes_regime": False,
    "fallback_changes_case_universe": False,
    "fallback_uses_all_prior": False,
    "fallback_is_active_personalization": False,
    "method_order": METHOD_ORDER,
    "methods": METHOD_LABELS,
    "selected_method_slug": PERSONALIZED_METHOD_SLUG,
    "selected_method": PERSONALIZED_METHOD_LABEL,
    "selection_source": PERSONALIZED_SELECTION_SOURCE,
    "selection_status": "selected_for_downstream_candidate_export",
    "winner_contract_version": WINNER_CONTRACT_VERSION,
    "stage1_winner_contract_version": stage1_winner.get("contract_version"),
    "stage1_winner_method_key": BASELINE_METHOD_SLUG,
    "stage1_winner_method_label": BASELINE_METHOD_LABEL,
    "stage1_winner_contract_path": str(STAGE1_WINNER_MANIFEST_PATH),
    "baseline_candidate_path": str(BASELINE_CANDIDATES_PATH),
    "evidence_scope": ITEM_EVIDENCE_SCOPE,
    "query_evidence_scope": "target_review_safe_signals_only",
    "personalization_evidence_scope": PROFILE_EVIDENCE_SCOPE,
    "dense_source": DENSE_TEXT_COLUMN,
    "sparse_source": SPARSE_TEXT_COLUMN,
    "profile_graph_source": "profile_safe_metadata_functional_and_brand_facets_from_qchs_selected_prior_items",
    "user_prior_enabled": True,
    "non_cold_stage1_profile_required": False,
    "non_cold_baseline_fallback_enabled": True,
    "cold_baseline_fallback_policy": "exact_query_only_winner_copy",
    "stage1_usable_prior_item_definition": "training_safe_prior_item_with_query_alignable_metadata_facets_and_profile_safe_functional_or_brand_facets",
    "non_cold_queries_with_zero_usable_stage1_prior_items": int((
        queries["regime"].ne("cold")
        & queries["stage1_usable_prior_item_count"].eq(0)
    ).sum()),
    "raw_prior_review_text_loaded": False,
    "prior_rating_loaded": False,
    "prior_sentiment_loaded": False,
    "raw_prior_brand_column_loaded": False,
    "brand_profile_enabled": True,
    "brand_profile_source": "Notebook04 profile-safe item facets joined by prior_item_id",
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "historical_review_reputation_enabled": True,
    "historical_review_reputation_used_as_profile_evidence": False,
    "target_item_metadata_used_for_profile": False,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "qchs_policy": (
        "IDF-weighted query-to-prior metadata functional-facet alignment with weights 0.70 token overlap, "
        "0.20 token-boundary exact phrase, and 0.10 matched role over metadata functional facets; "
        "top-12 prior items; softmax temperature 0.25; selected items contribute all profile-safe "
        "functional and brand facets; 16 anchors with four per role"
    ),
    "profile_sparse_qchs_policy": (
        "fuse the unchanged Notebook 07 query-only winner with BM25 retrieval from the repeated active query "
        "plus QCHS anchor tokens using RRF k=60 and weights [1.0, 0.8]"
    ),
    "profile_hybrid_qchs_policy": (
        "fuse the unchanged Notebook 07 query-only winner, dense_text retrieval from the QCHS-expanded query, "
        "and sparse_text QCHS retrieval using functional-plus-brand profile anchors and RRF k=60 "
        "with weights [1.0, 0.5, 0.7]"
    ),
    "profile_full_qchs_policy": (
        "fuse the unchanged Notebook 07 query-only winner, dense QCHS, sparse QCHS, and profile-safe "
        "functional-plus-brand graph expansion using RRF k=60 and weights [1.0, 0.4, 0.6, 0.6]"
    ),
    "output_paths": {
        "candidate_lists": str(CANDIDATE_LISTS_PATH),
        "per_query_metrics": str(PER_QUERY_METRICS_PATH),
        "profile_diagnostics": str(PROFILE_DIAGNOSTICS_PATH),
        "method_selection": str(METHOD_SELECTION_PATH),
        "run_manifest": str(MANIFEST_PATH),
        "winner_contract": str(WINNER_MANIFEST_PATH),
    },
}

winner_manifest = {
    "contract_version": WINNER_CONTRACT_VERSION,
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "stage": "stage1_personalized_retrieval",
    "selection_source": PERSONALIZED_SELECTION_SOURCE,
    "automatic_winner_method_slug": automatic_personalized_winner,
    "winner_method_slug": PERSONALIZED_METHOD_SLUG,
    "winner_method_label": PERSONALIZED_METHOD_LABEL,
    "selection_rank": int(personalized_winner_row["selection_rank"]),
    "selection_metric": selection_metric,
    "selection_metrics": {
        selection_metric: float(personalized_winner_row[selection_metric]),
        selection_ndcg: float(personalized_winner_row[selection_ndcg]),
        selection_mrr: float(personalized_winner_row[selection_mrr]),
        "regime_hit_rate_min": float(personalized_winner_row["regime_hit_rate_min"]),
        "regime_hit_rate_std": float(personalized_winner_row["regime_hit_rate_std"]),
        "online_personalized_runtime_sec": float(
            personalized_winner_row["online_personalized_runtime_sec"]
        ),
    },
    "selection_rule": str(personalized_winner_row["selection_rule"]),
    "query_only_winner_method_key": BASELINE_METHOD_SLUG,
    "query_only_winner_method_label": BASELINE_METHOD_LABEL,
    "query_only_winner_candidate_path": str(BASELINE_CANDIDATES_PATH),
    "query_only_winner_contract_path": str(STAGE1_WINNER_MANIFEST_PATH),
    "candidate_lists_path": str(CANDIDATE_LISTS_PATH),
    "candidate_list_filter": {
        "column": "method_slug",
        "value": PERSONALIZED_METHOD_SLUG,
    },
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": int(MAX_RETRIEVAL_K),
    "effective_candidate_count_per_query": int(EXPECTED_CANDIDATE_K),
    "catalog_size": int(catalog_size),
    "query_count": int(len(queries)),
    "exact_k_validation_passed": True,
    "user_prior_enabled": True,
    "query_only_retrieval_evidence_scope": ITEM_EVIDENCE_SCOPE,
    "personalization_evidence_scope": PROFILE_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": True,
    "historical_review_reputation_used_as_profile_evidence": False,
    "brand_profile_enabled": True,
    "brand_profile_source": "Notebook04 profile-safe item facets joined by prior_item_id",
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "method_selection_path": str(METHOD_SELECTION_PATH),
    "personalized_run_manifest_path": str(MANIFEST_PATH),
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, indent=2)
with open(WINNER_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(winner_manifest, file, indent=2)

required_outputs = [
    CANDIDATE_LISTS_PATH,
    PER_QUERY_METRICS_PATH,
    PROFILE_DIAGNOSTICS_PATH,
    METHOD_SELECTION_PATH,
    MANIFEST_PATH,
    WINNER_MANIFEST_PATH,
]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Missing Notebook 08 outputs: {missing_outputs}")

winner_check = load_json(WINNER_MANIFEST_PATH)
if winner_check.get("winner_method_slug") != PERSONALIZED_METHOD_SLUG:
    raise RuntimeError("Reloaded personalized winner contract method mismatch.")
if winner_check.get("query_only_winner_method_key") != BASELINE_METHOD_SLUG:
    raise RuntimeError("Reloaded personalized winner contract baseline mismatch.")
if winner_check.get("query_only_retrieval_evidence_scope") != ITEM_EVIDENCE_SCOPE:
    raise RuntimeError("Reloaded personalized winner contract evidence scope mismatch.")
if winner_check.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Reloaded personalized winner contract must enable item-side historical review signals.")
if winner_check.get("historical_review_reputation_used_as_profile_evidence") is not False:
    raise RuntimeError("Historical review signals must not be used as QCHS profile evidence.")
if winner_check.get("brand_profile_enabled") is not True:
    raise RuntimeError("Brand must be enabled in the QCHS profile source.")
if winner_check.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Brand must not be matched from the synthetic query.")
if winner_check.get("candidate_lists_path") != str(CANDIDATE_LISTS_PATH):
    raise RuntimeError("Reloaded personalized winner contract candidate-list path mismatch.")

print("Output:", CANDIDATE_LISTS_PATH)
print("Output:", PER_QUERY_METRICS_PATH)
print("Output:", METHOD_SELECTION_PATH)
print("Output:", MANIFEST_PATH)
print("Output:", WINNER_MANIFEST_PATH)
print("Rows: candidate lists", len(candidate_lists))
print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Selected personalized winner:", PERSONALIZED_METHOD_SLUG, "-", PERSONALIZED_METHOD_LABEL)
print("Validation: PASS")


Cases: 1968
QCHS-active cases: 553
Fallback cases: 1415
Non-cold no-alignment fallbacks: 759
Fallback rate: 0.719004
Fallback counts by regime: {'cold': 656, 'weak': 487, 'strong': 272}
Candidate fallback validation: passed
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/personalized_retrieval_candidate_lists_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/personalized_retrieval_per_query_metrics_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/personalized_retrieval_method_selection_herbal.csv
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/personalized_retrieval_manifest_herbal.json
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/person

## Supplementary Diagnostic — QCHS-Active Conditional NDCG@5

This diagnostic restricts the P1-minus-P0 comparison to cases classified as QCHS-active before any retrieval outcome is observed. Activity requires at least one selected prior item and at least one profile-safe facet. Candidate-list change is reported descriptively and is never used to define the subset.

Because every fallback case copies the query-only pool exactly, its P1-minus-P0 difference is zero. The coverage-weighted active-subset mean must therefore reconcile exactly with the full-sample mean. This conditional diagnostic does not replace the full-sample Stage-1 estimand or its headline Recall@1,000 endpoint.


In [16]:
# ==== Export QCHS Coverage and the Active-Subset Diagnostic ====
from pathlib import Path
import numpy as np
import pandas as pd

METRIC = "NDCG@5"
ZERO_TOL = 1e-12

# This supplementary cell uses the completed Notebook 08 objects.
# In a fresh session, load the canonical outputs before running this diagnostic.
required_objects = [
    "CATEGORY_ID", "OUTPUT_DIR", "REGIME_ORDER",
    "BASELINE_METHOD_SLUG", "PERSONALIZED_METHOD_SLUG",
    "per_query_metrics",
    "candidate_lists", "profile_diagnostics",
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(f"Run Notebook 08 completely first. Missing: {missing_objects}")

P0_METHOD = str(BASELINE_METHOD_SLUG)
P1_METHOD = str(PERSONALIZED_METHOD_SLUG)

OUTPUT_DIR = Path(OUTPUT_DIR)
ACTIVE_CASE_PATH = OUTPUT_DIR / f"qchs_active_case_metrics_{CATEGORY_ID}.parquet"
ACTIVE_SUMMARY_PATH = OUTPUT_DIR / f"qchs_active_conditional_summary_{CATEGORY_ID}.csv"
COVERAGE_PATH = OUTPUT_DIR / f"qchs_coverage_summary_{CATEGORY_ID}.csv"
FALLBACK_QC_PATH = OUTPUT_DIR / f"qchs_fallback_identity_qc_{CATEGORY_ID}.csv"


def require_columns(frame, columns, name):
    missing = [column for column in columns if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{name} is missing columns: {missing}")


def as_bool(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    return values.fillna("").astype(str).str.strip().str.lower().isin(
        {"1", "true", "t", "yes", "y"}
    )


def as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if hasattr(value, "tolist"):
        value = value.tolist()
        return value if isinstance(value, list) else [value]
    raise TypeError(f"Expected a list-like value, found {type(value)!r}")


def scopes(frame):
    yield "overall", "overall", frame
    yield "non_cold", "non_cold", frame.loc[frame["regime"].ne("cold")]
    observed = frame["regime"].dropna().astype(str).unique().tolist()
    for regime in dict.fromkeys([*list(REGIME_ORDER), *observed]):
        yield "regime", str(regime), frame.loc[frame["regime"].eq(regime)]


# Define QCHS activity without using retrieval outcomes.
profile_required = [
    "case_id", "user_id", "regime", "target_parent_asin",
    "qchs_selected_prior_item_count", "qchs_profile_safe_facet_count",
    "qchs_profile_available", "profile_fallback_flag",
]
require_columns(profile_diagnostics, profile_required, "profile_diagnostics")
profile = profile_diagnostics.copy()
for column in ["case_id", "user_id", "regime", "target_parent_asin"]:
    profile[column] = profile[column].astype(str)
if profile["case_id"].duplicated().any():
    raise RuntimeError("profile_diagnostics must contain one row per case_id.")

profile["qchs_selected_prior_item_count"] = pd.to_numeric(
    profile["qchs_selected_prior_item_count"], errors="raise"
).astype(int)
profile["qchs_profile_safe_facet_count"] = pd.to_numeric(
    profile["qchs_profile_safe_facet_count"], errors="raise"
).astype(int)

# Recompute the same availability rule used by the main pipeline.
profile["qchs_active"] = as_bool(profile["qchs_profile_available"])
recomputed_qchs_active = (
    profile["qchs_selected_prior_item_count"].gt(0)
    & profile["qchs_profile_safe_facet_count"].gt(0)
)
if not profile["qchs_active"].eq(recomputed_qchs_active).all():
    raise RuntimeError("qchs_profile_available disagrees with the recomputed QCHS-active contract.")
if not (~profile["qchs_active"]).eq(as_bool(profile["profile_fallback_flag"])).all():
    raise RuntimeError("profile_fallback_flag must be the inverse of qchs_active.")
if profile.loc[profile["regime"].eq("cold"), "qchs_active"].any():
    raise RuntimeError("Cold cases must not be QCHS-active.")


# Pair the selected personalized method with the query-only baseline on identical cases.
metric_required = [
    "case_id", "user_id", "regime", "target_parent_asin", "method_slug", "rank", METRIC,
]
require_columns(per_query_metrics, metric_required, "per_query_metrics")
metrics = per_query_metrics.copy()
for column in ["case_id", "user_id", "regime", "target_parent_asin", "method_slug"]:
    metrics[column] = metrics[column].astype(str)
keys = ["case_id", "user_id", "regime", "target_parent_asin"]
p0 = metrics.loc[metrics["method_slug"].eq(P0_METHOD), [*keys, "rank", METRIC]].rename(
    columns={"rank": "p0_rank", METRIC: "p0_ndcg_at_5"}
)
p1 = metrics.loc[metrics["method_slug"].eq(P1_METHOD), [*keys, "rank", METRIC]].rename(
    columns={"rank": "p1_rank", METRIC: "p1_ndcg_at_5"}
)
if p0["case_id"].duplicated().any() or p1["case_id"].duplicated().any():
    raise RuntimeError("P0 and P1 must each contain one metric row per case_id.")
paired = p0.merge(p1, on=keys, validate="one_to_one")
if set(paired["case_id"]) != set(profile["case_id"]):
    raise RuntimeError("Paired P0/P1 metrics do not cover the profile case universe.")

profile_optional = [
    "prior_review_event_count", "prior_unique_item_count", "raw_prior_item_count",
    "training_safe_prior_item_count", "prior_depth_bin", "stage1_usable_prior_item_count",
    "stage1_profile_safe_prior_item_count", "qchs_selected_ratio", "qchs_alignment_mean",
    "qchs_alignment_max", "qchs_attention_entropy", "qchs_brand_facet_count",
    "qchs_functional_facet_count", "qchs_anchor_phrase_count", "profile_fallback_reason",
]
profile_columns = [
    *keys, "qchs_selected_prior_item_count", "qchs_profile_safe_facet_count", "qchs_active",
    *[column for column in profile_optional if column in profile.columns],
]
paired = paired.merge(profile[profile_columns], on=keys, validate="one_to_one")
paired["p0_rank"] = pd.to_numeric(paired["p0_rank"], errors="raise").astype(int)
paired["p1_rank"] = pd.to_numeric(paired["p1_rank"], errors="raise").astype(int)
paired["p0_ndcg_at_5"] = pd.to_numeric(paired["p0_ndcg_at_5"], errors="raise")
paired["p1_ndcg_at_5"] = pd.to_numeric(paired["p1_ndcg_at_5"], errors="raise")
paired["delta_ndcg_at_5"] = paired["p1_ndcg_at_5"] - paired["p0_ndcg_at_5"]
fallback_delta = paired.loc[~paired["qchs_active"], "delta_ndcg_at_5"]
if not np.allclose(fallback_delta.to_numpy(float), 0.0, rtol=0.0, atol=ZERO_TOL):
    raise RuntimeError(
        "Fallback cases must have zero P1-P0 NDCG@5 delta before ITT reconciliation."
    )
paired["target_newly_entered_pool"] = paired["p0_rank"].eq(0) & paired["p1_rank"].gt(0)
paired["target_newly_lost_pool"] = paired["p0_rank"].gt(0) & paired["p1_rank"].eq(0)


# Verify exact fallback identity; candidate-list changes are descriptive only.
candidate_required = [
    "case_id", "method_slug", "candidate_parent_asin_list", "candidate_score_list",
    "profile_fallback_flag",
]
require_columns(candidate_lists, candidate_required, "candidate_lists")
candidates = candidate_lists.copy()
candidates["case_id"] = candidates["case_id"].astype(str)
candidates["method_slug"] = candidates["method_slug"].astype(str)
c0 = candidates.loc[candidates["method_slug"].eq(P0_METHOD), [
    "case_id", "candidate_parent_asin_list", "candidate_score_list"
]].rename(columns={"candidate_parent_asin_list": "p0_items", "candidate_score_list": "p0_scores"})
c1 = candidates.loc[candidates["method_slug"].eq(P1_METHOD), [
    "case_id", "candidate_parent_asin_list", "candidate_score_list", "profile_fallback_flag"
]].rename(columns={
    "candidate_parent_asin_list": "p1_items", "candidate_score_list": "p1_scores",
    "profile_fallback_flag": "p1_fallback",
})
candidate_qc = c0.merge(c1, on="case_id", validate="one_to_one").merge(
    profile[["case_id", "regime", "qchs_active"]], on="case_id", validate="one_to_one"
)
if set(candidate_qc["case_id"]) != set(profile["case_id"]):
    raise RuntimeError("P0/P1 candidate rows do not cover the profile case universe.")
if not as_bool(candidate_qc["p1_fallback"]).eq(~candidate_qc["qchs_active"]).all():
    raise RuntimeError("Selected P1 fallback flags disagree with qchs_active.")

candidate_qc["candidate_id_order_identical"] = [
    as_list(left) == as_list(right) for left, right in zip(candidate_qc["p0_items"], candidate_qc["p1_items"])
]
candidate_qc["candidate_scores_identical"] = [
    bool(
        np.asarray(as_list(left), dtype=float).shape == np.asarray(as_list(right), dtype=float).shape
        and np.allclose(
            np.asarray(as_list(left), dtype=float), np.asarray(as_list(right), dtype=float),
            rtol=0.0, atol=0.0, equal_nan=True,
        )
    )
    for left, right in zip(candidate_qc["p0_scores"], candidate_qc["p1_scores"])
]
candidate_qc["candidate_list_changed"] = ~candidate_qc["candidate_id_order_identical"]
paired = paired.merge(
    candidate_qc[["case_id", "candidate_id_order_identical", "candidate_scores_identical", "candidate_list_changed"]],
    on="case_id", validate="one_to_one",
)

fallback_qc_rows = []
for scope_type, scope_value, group in scopes(candidate_qc):
    fallback = group.loc[~group["qchs_active"]]
    identity_pass = bool(
        fallback["candidate_id_order_identical"].all()
        and fallback["candidate_scores_identical"].all()
    ) if len(fallback) else True
    fallback_qc_rows.append({
        "category_id": CATEGORY_ID, "scope_type": scope_type, "scope_value": scope_value,
        "fallback_case_count": int(len(fallback)),
        "candidate_id_order_identity_count": int(fallback["candidate_id_order_identical"].sum()),
        "candidate_score_identity_count": int(fallback["candidate_scores_identical"].sum()),
        "fallback_identity_pass": identity_pass,
    })
qchs_fallback_identity_qc = pd.DataFrame(fallback_qc_rows)
if not qchs_fallback_identity_qc["fallback_identity_pass"].all():
    qchs_fallback_identity_qc.to_csv(FALLBACK_QC_PATH, index=False, encoding="utf-8-sig")
    raise RuntimeError("Fallback identity failed; QC was exported. Do not use the active-only result yet.")


# Export coverage and active-subset NDCG@5 diagnostics with full-sample reconciliation.
profile_qc = profile.merge(
    candidate_qc[["case_id", "candidate_list_changed"]], on="case_id", validate="one_to_one"
)
coverage_rows, summary_rows = [], []
for scope_type, scope_value, group in scopes(profile_qc):
    active = group.loc[group["qchs_active"]]
    n_total, n_active = len(group), len(active)
    coverage_rows.append({
        "category_id": CATEGORY_ID, "baseline_method_slug": P0_METHOD,
        "personalized_method_slug": P1_METHOD, "scope_type": scope_type,
        "scope_value": scope_value, "total_case_count": int(n_total),
        "qchs_active_case_count": int(n_active), "fallback_case_count": int(n_total - n_active),
        "qchs_active_rate": float(n_active / n_total) if n_total else np.nan,
        "fallback_rate": float((n_total - n_active) / n_total) if n_total else np.nan,
        "active_candidate_list_changed_count": int(active["candidate_list_changed"].sum()),
        "active_candidate_list_changed_rate": float(active["candidate_list_changed"].mean()) if n_active else np.nan,
        "activation_definition": "qchs_selected_prior_item_count > 0 and qchs_profile_safe_facet_count > 0",
    })

for scope_type, scope_value, all_cases in scopes(paired):
    active = all_cases.loc[all_cases["qchs_active"]]
    n_total, n_active = len(all_cases), len(active)
    active_rate = float(n_active / n_total) if n_total else np.nan
    delta = active["delta_ndcg_at_5"].to_numpy(float)
    full_delta = float(all_cases["delta_ndcg_at_5"].mean()) if n_total else np.nan
    active_delta = float(delta.mean()) if n_active else np.nan
    weighted_delta = float(active_rate * active_delta) if n_active else (0.0 if n_total else np.nan)
    positive = int((delta > ZERO_TOL).sum())
    negative = int((delta < -ZERO_TOL).sum())
    tie = int(n_active - positive - negative)
    summary_rows.append({
        "category_id": CATEGORY_ID, "baseline_method_slug": P0_METHOD,
        "personalized_method_slug": P1_METHOD, "scope_type": scope_type,
        "scope_value": scope_value, "metric": METRIC, "total_case_count": int(n_total),
        "qchs_active_case_count": int(n_active), "qchs_active_rate": active_rate,
        "active_baseline_ndcg_at_5": float(active["p0_ndcg_at_5"].mean()) if n_active else np.nan,
        "active_personalized_ndcg_at_5": float(active["p1_ndcg_at_5"].mean()) if n_active else np.nan,
        "active_mean_delta_ndcg_at_5": active_delta,
        "active_median_delta_ndcg_at_5": float(np.median(delta)) if n_active else np.nan,
        "active_positive_delta_count": positive, "active_tie_count": tie,
        "active_negative_delta_count": negative,
        "active_positive_delta_rate": float(positive / n_active) if n_active else np.nan,
        "active_tie_rate": float(tie / n_active) if n_active else np.nan,
        "active_negative_delta_rate": float(negative / n_active) if n_active else np.nan,
        "active_candidate_list_changed_count": int(active["candidate_list_changed"].sum()),
        "active_candidate_list_changed_rate": float(active["candidate_list_changed"].mean()) if n_active else np.nan,
        "active_target_newly_entered_pool_count": int(active["target_newly_entered_pool"].sum()),
        "active_target_newly_lost_pool_count": int(active["target_newly_lost_pool"].sum()),
        "canonical_full_sample_mean_delta_ndcg_at_5": full_delta,
        "coverage_weighted_active_mean_delta_ndcg_at_5": weighted_delta,
        "itt_reconciliation_absolute_error": abs(full_delta - weighted_delta) if n_total else np.nan,
        "estimand_label": "conditional_descriptive_uplift_among_pre_outcome_qchs_active_cases",
    })

qchs_coverage_summary = pd.DataFrame(coverage_rows)
qchs_active_conditional_summary = pd.DataFrame(summary_rows)
max_error = float(qchs_active_conditional_summary["itt_reconciliation_absolute_error"].fillna(0).max())
if max_error > ZERO_TOL:
    raise RuntimeError(f"ITT reconciliation failed; maximum error={max_error}")

qchs_active_case_metrics = paired.loc[paired["qchs_active"]].copy()
qchs_active_case_metrics.insert(0, "category_id", CATEGORY_ID)
qchs_active_case_metrics.insert(1, "baseline_method_slug", P0_METHOD)
qchs_active_case_metrics.insert(2, "personalized_method_slug", P1_METHOD)
qchs_active_case_metrics["activation_definition"] = (
    "qchs_selected_prior_item_count > 0 and qchs_profile_safe_facet_count > 0"
)
qchs_active_case_metrics["selection_uses_candidate_outcome"] = False

qchs_active_case_metrics.to_parquet(ACTIVE_CASE_PATH, index=False)
qchs_active_conditional_summary.to_csv(ACTIVE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
qchs_coverage_summary.to_csv(COVERAGE_PATH, index=False, encoding="utf-8-sig")
qchs_fallback_identity_qc.to_csv(FALLBACK_QC_PATH, index=False, encoding="utf-8-sig")

print("Canonical outputs modified: False")
print("P0 / P1:", P0_METHOD, "/", P1_METHOD)
print("Total / active / fallback:", len(profile), len(qchs_active_case_metrics), int((~profile["qchs_active"]).sum()))
print("Maximum ITT reconciliation error:", max_error)
for path in [ACTIVE_CASE_PATH, ACTIVE_SUMMARY_PATH, COVERAGE_PATH, FALLBACK_QC_PATH]:
    print("Output:", path)
display(qchs_coverage_summary)
display(qchs_active_conditional_summary)
display(qchs_fallback_identity_qc)

Canonical outputs modified: False
P0 / P1: hybrid_dense_bm25 / profile_sparse_qchs
Total / active / fallback: 1968 553 1415
Maximum ITT reconciliation error: 2.710505431213761e-20
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/qchs_active_case_metrics_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/qchs_active_conditional_summary_herbal.csv
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/qchs_coverage_summary_herbal.csv
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/qchs_fallback_identity_qc_herbal.csv


,category_id,baseline_method_slug,personalized_method_slug,scope_type,scope_value,total_case_count,qchs_active_case_count,fallback_case_count,qchs_active_rate,fallback_rate,active_candidate_list_changed_count,active_candidate_list_changed_rate,activation_definition
0,herbal,hybrid_dense_bm25,profile_sparse_qchs,overall,overall,1968,553,1415,0.280996,0.719004,553,1.0,qchs_selected_prior_item_count > 0 and qchs_pr...
1,herbal,hybrid_dense_bm25,profile_sparse_qchs,non_cold,non_cold,1312,553,759,0.421494,0.578506,553,1.0,qchs_selected_prior_item_count > 0 and qchs_pr...
2,herbal,hybrid_dense_bm25,profile_sparse_qchs,regime,cold,656,0,656,0.000000,1.000000,0,NaN,qchs_selected_prior_item_count > 0 and qchs_pr...
3,herbal,hybrid_dense_bm25,profile_sparse_qchs,regime,weak,656,169,487,0.257622,0.742378,169,1.0,qchs_selected_prior_item_count > 0 and qchs_pr...
4,herbal,hybrid_dense_bm25,profile_sparse_qchs,regime,strong,656,384,272,0.585366,0.414634,384,1.0,qchs_selected_prior_item_count > 0 and qchs_pr...


,category_id,baseline_method_slug,personalized_method_slug,scope_type,scope_value,metric,total_case_count,qchs_active_case_count,qchs_active_rate,active_baseline_ndcg_at_5,...,active_tie_rate,active_negative_delta_rate,active_candidate_list_changed_count,active_candidate_list_changed_rate,active_target_newly_entered_pool_count,active_target_newly_lost_pool_count,canonical_full_sample_mean_delta_ndcg_at_5,coverage_weighted_active_mean_delta_ndcg_at_5,itt_reconciliation_absolute_error,estimand_label
0,herbal,hybrid_dense_bm25,profile_sparse_qchs,overall,overall,NDCG@5,1968,553,0.280996,0.030017,...,0.963834,0.018083,553,1.0,33,12,0.000172,0.000172,2.710505e-20,conditional_descriptive_uplift_among_pre_outco...
1,herbal,hybrid_dense_bm25,profile_sparse_qchs,non_cold,non_cold,NDCG@5,1312,553,0.421494,0.030017,...,0.963834,0.018083,553,1.0,33,12,0.000258,0.000258,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
2,herbal,hybrid_dense_bm25,profile_sparse_qchs,regime,cold,NDCG@5,656,0,0.000000,NaN,...,NaN,NaN,0,NaN,0,0,0.000000,0.000000,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
3,herbal,hybrid_dense_bm25,profile_sparse_qchs,regime,weak,NDCG@5,656,169,0.257622,0.007796,...,0.970414,0.011834,169,1.0,20,6,0.000722,0.000722,0.000000e+00,conditional_descriptive_uplift_among_pre_outco...
4,herbal,hybrid_dense_bm25,profile_sparse_qchs,regime,strong,NDCG@5,656,384,0.585366,0.039797,...,0.960938,0.020833,384,1.0,13,6,-0.000206,-0.000206,2.710505e-20,conditional_descriptive_uplift_among_pre_outco...


,category_id,scope_type,scope_value,fallback_case_count,candidate_id_order_identity_count,candidate_score_identity_count,fallback_identity_pass
0,herbal,overall,overall,1415,1415,1415,True
1,herbal,non_cold,non_cold,759,759,759,True
2,herbal,regime,cold,656,656,656,True
3,herbal,regime,weak,487,487,487,True
4,herbal,regime,strong,272,272,272,True
